# Paper 2 — Goal 3
## Notebook 36 — Final robustness package and computational completion

This notebook starts **after Goal 3 Stage E has passed**.

It does **not** rerun the 14,792 waveform measurements and does **not** alter the
frozen perturbation grid. The Stage-E response table is the immutable controlled
measurement input.

The notebook closes the remaining prespecified Goal-3 analysis work:

1. hard audit of Stage-E target-Q ordering, inferential status, support behavior,
   and prediction availability;
2. exemplar-level sensitivity versus the exemplar-averaged primary estimand;
3. ordered-factor / heterogeneity / QDIST sensitivity consolidation;
4. a governed fixed-baseline-segmentation feasibility audit;
5. reconstruction of the already-frozen Goal-2 HGB sensitivity models for
   **outer repeat 1 only**, verification against the original Goal-2 HGB OOF
   checkpoints, and application to the already-measured Goal-3 perturbation rows;
6. paired 2,000-participant HGB-versus-ridge perturbation-slope bootstrap;
7. a machine-readable Goal-3 computational completion manifest.

### Important interpretation boundary

Natural Q–A localization remains observational. The same-source controlled
experiment supports **direct sensitivity to the imposed digital transformation**
only. It does not establish that an analogous natural-data association was caused
by that physical mechanism.

### Important provenance rule for HGB

No HGB hyperparameter is selected in this notebook. The HGB structures,
iteration counts, residualizer penalties, folds, and training rows were all frozen
in Goal 2 before Goal-3 controlled clinical responses were examined. This notebook
reconstructs only the already-selected first-repeat HGB states from unmodified
outer-training data and first proves that they reproduce the original Goal-2 HGB
OOF predictions.

### Run instruction

Use **Kernel → Restart Kernel and Run All Cells**.

The expected final state is:

`GOAL 3 COMPUTATIONAL COMPLETION: PASS_PENDING_PUBLICATION_FIGURES`

Publication figures are intentionally handled in the next **figure-only**
notebook so the visual package cannot change any analysis result.

In [1]:
from __future__ import annotations

import ast as pyast
import hashlib
import json
import math
import os
import platform
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    HistGradientBoostingRegressor,
)

BASE_SEED = 20260825
N_BOOTSTRAPS = 2000
OUTER_REPEAT = 1
OUTER_FOLDS = 5

RIDGE_MODELS = ["M_A", "M_A+Q", "M_A-resQ"]
HGB_MODELS = ["HGB_M_A", "HGB_M_A+Q", "HGB_M_A-resQ"]
TASKS = ["diagnosis", "severity"]

ENGINE_VERSION = "goal3-completion-v1.0.0"

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(path: Path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def stable_hash(payload):
    text = json.dumps(
        payload,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def deterministic_seed(*tokens, modulus=1_000_000):
    digest = stable_hash(list(tokens))
    return BASE_SEED + (int(digest[:16], 16) % modulus)

def atomic_csv(frame: pd.DataFrame, path: Path, *, allow_empty=False):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not isinstance(frame, pd.DataFrame):
        raise TypeError(f"{path.name}: expected DataFrame.")
    if frame.empty and not allow_empty:
        raise ValueError(f"{path.name}: refusing to write an empty table.")
    tmp = path.with_name("." + path.name + ".tmp")
    frame.to_csv(tmp, index=False)
    if not tmp.exists() or tmp.stat().st_size == 0:
        raise IOError(f"{path.name}: temporary output is empty.")
    os.replace(tmp, path)

def atomic_json(payload: dict, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name("." + path.name + ".tmp")
    tmp.write_text(
        json.dumps(payload, indent=2, sort_keys=True, default=str),
        encoding="utf-8",
    )
    os.replace(tmp, path)

def safe_csv(path: Path, required=None, *, allow_empty=False):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing/empty CSV: {path}")
    frame = pd.read_csv(path, low_memory=False)
    if frame.empty and not allow_empty:
        raise ValueError(f"{path.name}: no rows.")
    if required:
        missing = set(required) - set(frame.columns)
        if missing:
            raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    return frame

def find_root():
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        p = Path(override).expanduser().resolve()
        if (p / "outputs" / "goal3").exists():
            return p
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {p}")

    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (
            (p / "outputs" / "goal3").exists()
            and (p / "outputs" / "goal2").exists()
            and (p / "data" / "processed").exists()
        ):
            return p
    raise FileNotFoundError(
        "Could not locate Paper_2_Leakage/Code. "
        "Run from the repository or set PAPER2_ROOT."
    )

ROOT = find_root()

STAGE_A_CANDIDATES = [
    ROOT / "outputs" / "goal3" / "stageA_v1_1",
    ROOT / "outputs" / "goal3" / "stageA_v1_0",
]
STAGE_A = next(
    (p for p in STAGE_A_CANDIDATES if (p / "SUCCESS_STAGE_A.json").exists()),
    None,
)
if STAGE_A is None:
    raise FileNotFoundError("No successful Goal-3 Stage-A directory found.")

STAGE_E = (
    ROOT / "outputs" / "goal3"
    / "stageE_controlled_perturbation_v1_0" / "final"
)
STAGE_E_TABLES = STAGE_E / "tables"
STAGE_E_AUDIT = STAGE_E / "audit"

GOAL2 = (
    ROOT / "outputs" / "goal2"
    / "goal2_completion_v1_0" / "final"
)

OUT = (
    ROOT / "outputs" / "goal3"
    / "goal3_completion_v1_0" / "final"
)
TABLES = OUT / "tables"
AUDIT = OUT / "audit"
HGB_DIR = OUT / "hgb_states"
for d in [OUT, TABLES, AUDIT, HGB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Paper 2 root:", ROOT)
print("Stage A:", STAGE_A)
print("Stage E:", STAGE_E)
print("Completion output:", OUT)

Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Stage A: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageA_v1_0
Stage E: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageE_controlled_perturbation_v1_0\final
Completion output: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\goal3_completion_v1_0\final


## 1. Hard Stage-E completion gate

This cell treats Stage E as immutable input. It verifies the full 14,792-row
controlled response and the primary inferential package before any supplementary
analysis begins.

In [3]:
stagee_seal_path = (
    STAGE_E
    / "GOAL3_STAGE_E_PRIMARY_SEAL.json"
)

if not stagee_seal_path.exists():
    raise FileNotFoundError(
        stagee_seal_path
    )


stagee_seal = json.loads(
    stagee_seal_path.read_text(
        encoding="utf-8"
    )
)

if stagee_seal.get("status") != "PASS":
    raise RuntimeError(
        "Stage E is not PASS: "
        f"{stagee_seal.get('status')!r}"
    )


# =====================================================================
# LOAD IMMUTABLE STAGE-E OUTPUTS
# =====================================================================

response = safe_csv(
    STAGE_E_TABLES
    / "goal3_controlled_response_exemplar.csv",
    required=[
        "task",
        "participant_id",
        "logical_recording_id",
        "outer_fold",
        "family",
        "transform",
        "dose_label",
        "dose_code",
        "exemplar",
        "execution_status",
        "age_at_recording_years",
        "y",
    ],
)


prediction_primary = safe_csv(
    STAGE_E_TABLES
    / "goal3_prediction_delta.csv",
    required=[
        "task",
        "participant_id",
        "transform",
        "dose_code",
        "model",
        "y",
        "delta_logit",
        "delta_prediction",
        "delta_error",
    ],
)


prediction_exemplar = safe_csv(
    STAGE_E_TABLES
    / "goal3_prediction_delta_exemplar.csv",
    required=[
        "task",
        "participant_id",
        "transform",
        "dose_code",
        "exemplar",
        "model",
        "y",
        "delta_logit",
        "delta_prediction",
        "delta_error",
    ],
)


feature_exemplar = safe_csv(
    STAGE_E_TABLES
    / "goal3_feature_delta_exemplar.csv",
    required=[
        "task",
        "participant_id",
        "transform",
        "dose_code",
        "dose_label",
        "exemplar",
        "measurement_type",
        "feature",
        "delta",
    ],
)


target_q = safe_csv(
    STAGE_E_TABLES
    / "goal3_target_q_verification.csv",
)


dose_tests = safe_csv(
    STAGE_E_TABLES
    / "goal3_dose_response_tests.csv",
    required=[
        "test_type",
        "task",
        "transform",
        "model",
        "endpoint",
        "estimate",
        "se",
        "p_value",
        "participants",
        "status",
    ],
)


bootstrap = safe_csv(
    STAGE_E_TABLES
    / "goal3_bootstrap_ci.csv",
    required=[
        "n_bootstraps",
        "status",
    ],
)


qdist_sens = safe_csv(
    STAGE_E_TABLES
    / "goal3_qdist_baseline_negative_sensitivity.csv",
    required=[
        "task",
        "model",
        "endpoint",
        "participants",
        "estimate_per_dose",
        "se",
        "p_value",
        "status",
    ],
)


offtarget_q = safe_csv(
    STAGE_E_TABLES
    / "goal3_offtarget_q_summary.csv",
    allow_empty=True,
)


support_failure = safe_csv(
    STAGE_E_TABLES
    / "goal3_support_failure_summary.csv",
)


# IMPORTANT:
# These are the actual column names produced by Notebook 35.
target_order = safe_csv(
    STAGE_E_AUDIT
    / "goal3_target_q_order_audit.csv",
    required=[
        "strict_low_medium_high_order",
        "all_medians_expected_direction",
    ],
)


# =====================================================================
# HARD STAGE-E CONTRACT CHECKS
# =====================================================================

if len(response) != 14_792:
    raise RuntimeError(
        "Expected 14,792 Stage-E rows; "
        f"found {len(response)}."
    )


measurement_unavailable = int(
    response[
        "execution_status"
    ]
    .ne("PASS")
    .sum()
)

if measurement_unavailable != 0:
    raise RuntimeError(
        "Stage E contains "
        f"{measurement_unavailable} "
        "measurement-unavailable rows."
    )


# This is the actual Stage-E ordering field.
target_all_ordered = bool(
    target_order[
        "strict_low_medium_high_order"
    ]
    .astype(bool)
    .all()
)

if not target_all_ordered:
    bad = target_order.loc[
        ~target_order[
            "strict_low_medium_high_order"
        ].astype(bool)
    ].copy()

    display(bad)

    raise RuntimeError(
        "At least one target-Q branch "
        "is not strictly ordered "
        "low < medium < high."
    )


target_all_expected = bool(
    target_order[
        "all_medians_expected_direction"
    ]
    .astype(bool)
    .all()
)

if not target_all_expected:
    bad = target_order.loc[
        ~target_order[
            "all_medians_expected_direction"
        ].astype(bool)
    ].copy()

    display(bad)

    raise RuntimeError(
        "At least one target-Q branch "
        "has medians in the wrong direction."
    )


# =====================================================================
# CROSS-CHECK AGAINST THE STAGE-E SEAL
# =====================================================================

seal_ordered = stagee_seal.get(
    "target_q_full_sample_all_strictly_ordered"
)

seal_expected = stagee_seal.get(
    "target_q_full_sample_all_expected_direction"
)

if seal_ordered is not True:
    raise RuntimeError(
        "Stage-E seal does not certify "
        "strict target-Q ordering."
    )

if seal_expected is not True:
    raise RuntimeError(
        "Stage-E seal does not certify "
        "expected-direction target-Q movement."
    )


# =====================================================================
# GEE PACKAGE
# =====================================================================

gee_bad = dose_tests.loc[
    ~dose_tests[
        "status"
    ]
    .astype(str)
    .eq("PASS")
].copy()

if len(gee_bad):
    display(gee_bad)

    raise RuntimeError(
        "At least one Stage-E GEE row "
        "is not PASS."
    )


# =====================================================================
# 2,000-PARTICIPANT BOOTSTRAP PACKAGE
# =====================================================================

bootstrap_bad = bootstrap.loc[
    ~bootstrap[
        "status"
    ]
    .astype(str)
    .eq("PASS")
].copy()

if len(bootstrap_bad):
    display(bootstrap_bad)

    raise RuntimeError(
        "At least one Stage-E bootstrap "
        "contrast is not PASS."
    )


if not (
    pd.to_numeric(
        bootstrap[
            "n_bootstraps"
        ],
        errors="coerce",
    )
    .eq(2000)
    .all()
):
    bad = bootstrap.loc[
        ~pd.to_numeric(
            bootstrap[
                "n_bootstraps"
            ],
            errors="coerce",
        )
        .eq(2000)
    ].copy()

    display(bad)

    raise RuntimeError(
        "Stage-E bootstrap package "
        "is not uniformly B=2,000."
    )


# =====================================================================
# QDIST BASELINE-NEGATIVE SENSITIVITY
# =====================================================================

qdist_bad = qdist_sens.loc[
    ~qdist_sens[
        "status"
    ]
    .astype(str)
    .eq("PASS")
].copy()

if len(qdist_bad):
    display(qdist_bad)

    raise RuntimeError(
        "QDIST baseline-negative "
        "sensitivity is incomplete."
    )


# =====================================================================
# FINAL REPORT
# =====================================================================

print(
    "=" * 78
)

print(
    "STAGE-E IMMUTABLE INPUT GATE: PASS"
)

print(
    "=" * 78
)

print(
    "Response rows:",
    len(response),
)

print(
    "Measurement-unavailable rows:",
    measurement_unavailable,
)

print(
    "GEE rows:",
    len(dose_tests),
)

print(
    "Bootstrap rows:",
    len(bootstrap),
)

print(
    "QDIST sensitivity rows:",
    len(qdist_sens),
)

print(
    "Target-Q strictly ordered "
    "in all task/transform branches:",
    target_all_ordered,
)

print(
    "Target-Q medians all in "
    "expected direction:",
    target_all_expected,
)

STAGE-E IMMUTABLE INPUT GATE: PASS
Response rows: 14792
Measurement-unavailable rows: 0
GEE rows: 720
Bootstrap rows: 48
QDIST sensitivity rows: 6
Target-Q strictly ordered in all task/transform branches: True
Target-Q medians all in expected direction: True


## 2. Prediction-availability and support audit

Perturbation-induced inference unavailability is preserved as measurement/model
behavior. No row is repaired by refitting, source substitution, or post-hoc
imputation outside the frozen model contract.

A baseline prediction failure would instead indicate a broken upstream bridge and
therefore fails this notebook.

In [4]:
prediction_availability_rows = []
prediction_unavailable_detail = []

for model in RIDGE_MODELS:
    status_col = f"prediction_status__{model}"
    error_col = f"prediction_error__{model}"

    if status_col not in response.columns:
        raise RuntimeError(f"Missing Stage-E status column: {status_col}")

    base_bad = response.loc[
        response["transform"].eq("baseline")
        & response[status_col].fillna("").astype(str).ne("PASS")
    ].copy()

    if len(base_bad):
        display(base_bad[
            ["task", "participant_id", "logical_recording_id", status_col, error_col]
        ])
        raise RuntimeError(
            f"{model}: unavailable prediction on unmodified baseline."
        )

    pert_bad = response.loc[
        ~response["transform"].eq("baseline")
        & response[status_col].fillna("").astype(str).ne("PASS")
    ].copy()

    prediction_availability_rows.append({
        "model": model,
        "baseline_unavailable_rows": len(base_bad),
        "perturbed_unavailable_rows": len(pert_bad),
        "perturbed_unavailable_participants": pert_bad["participant_id"].nunique(),
    })

    if len(pert_bad):
        z = pert_bad[
            [
                "task", "participant_id", "logical_recording_id", "outer_fold",
                "family", "transform", "dose_label", "dose_code", "exemplar",
                status_col, error_col,
            ]
        ].copy()
        z["model"] = model
        z = z.rename(columns={
            status_col: "prediction_status",
            error_col: "prediction_error",
        })
        prediction_unavailable_detail.append(z)

prediction_availability = pd.DataFrame(prediction_availability_rows)
prediction_unavailable_detail = (
    pd.concat(prediction_unavailable_detail, ignore_index=True)
    if prediction_unavailable_detail
    else pd.DataFrame()
)

if len(prediction_unavailable_detail):
    prediction_unavailable_summary = (
        prediction_unavailable_detail.groupby(
            [
                "task", "family", "transform", "dose_label",
                "dose_code", "model", "prediction_error",
            ],
            dropna=False,
            as_index=False,
        )
        .agg(
            unavailable_rows=("participant_id", "size"),
            unavailable_participants=("participant_id", "nunique"),
        )
    )
else:
    prediction_unavailable_summary = pd.DataFrame(
        columns=[
            "task", "family", "transform", "dose_label", "dose_code",
            "model", "prediction_error",
            "unavailable_rows", "unavailable_participants",
        ]
    )

atomic_csv(
    prediction_availability,
    TABLES / "goal3_prediction_availability_by_model.csv",
)
atomic_csv(
    prediction_unavailable_detail,
    TABLES / "goal3_prediction_unavailable_detail.csv",
    allow_empty=True,
)
atomic_csv(
    prediction_unavailable_summary,
    TABLES / "goal3_prediction_unavailable_summary.csv",
    allow_empty=True,
)

print("RIDGE PREDICTION AVAILABILITY")
display(prediction_availability)

if len(prediction_unavailable_summary):
    print("\nUNAVAILABLE-PREDICTION BREAKDOWN")
    display(
        prediction_unavailable_summary.sort_values(
            ["model", "task", "transform", "dose_code"]
        )
    )

RIDGE PREDICTION AVAILABILITY


,model,baseline_unavailable_rows,perturbed_unavailable_rows,perturbed_unavailable_participants
0,M_A,0,0,0
1,M_A+Q,0,191,15
2,M_A-resQ,0,191,15



UNAVAILABLE-PREDICTION BREAKDOWN


,task,family,transform,dose_label,dose_code,model,prediction_error,unavailable_rows,unavailable_participants
22,diagnosis,QREV,RIR_convolution_RMS_matched,low,1,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,5,4
24,diagnosis,QREV,RIR_convolution_RMS_matched,medium,2,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,10,7
20,diagnosis,QREV,RIR_convolution_RMS_matched,high,3,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,27,14
14,diagnosis,QGAIN,smooth_time_varying_gain,low,1,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,1,1
16,diagnosis,QGAIN,smooth_time_varying_gain,medium,2,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,2,2
12,diagnosis,QGAIN,smooth_time_varying_gain,high,3,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,5,5
2,diagnosis,QADD,stationary_colored_broadband,low,1,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,5,1
4,diagnosis,QADD,stationary_colored_broadband,medium,2,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,7,2
0,diagnosis,QADD,stationary_colored_broadband,high,3,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,15,4
10,diagnosis,QDIST,symmetric_hard_clipping,high,3,M_A+Q,RuntimeError: Fold-safe QCHAN merge produced m...,1,1


## 3. Consolidate prespecified primary/sensitivity inference

The numeric dose trend is primary. The ordered-factor rows, diagnosis class
interaction, severity-by-baseline-bulbar interaction, and clipping restriction
are preserved as secondary/sensitivity evidence.

In [5]:
expected_test_types = {
    "acoustic_dose_linear",
    "target_q_dose_linear",
    "clinical_dose_linear",
    "clinical_dose_factor",
    "dose_by_model_interaction",
    "diagnosis_dose_by_true_class",
    "severity_dose_by_baseline_bulbar",
}
observed_types = set(dose_tests["test_type"].astype(str))
missing_types = expected_test_types - observed_types
if missing_types:
    raise RuntimeError(
        f"Stage-E dose-response package missing test types: {sorted(missing_types)}"
    )

clinical_linear = dose_tests.loc[
    dose_tests["test_type"].eq("clinical_dose_linear")
].copy()

ordered_factor = dose_tests.loc[
    dose_tests["test_type"].eq("clinical_dose_factor")
].copy()

heterogeneity = dose_tests.loc[
    dose_tests["test_type"].isin([
        "diagnosis_dose_by_true_class",
        "severity_dose_by_baseline_bulbar",
    ])
].copy()

model_interactions = dose_tests.loc[
    dose_tests["test_type"].eq("dose_by_model_interaction")
].copy()

atomic_csv(
    clinical_linear,
    TABLES / "goal3_primary_clinical_dose_linear.csv",
)
atomic_csv(
    ordered_factor,
    TABLES / "goal3_ordered_factor_sensitivity.csv",
)
atomic_csv(
    heterogeneity,
    TABLES / "goal3_heterogeneity_sensitivity.csv",
)
atomic_csv(
    model_interactions,
    TABLES / "goal3_model_interaction_tests.csv",
)
atomic_csv(
    qdist_sens,
    TABLES / "goal3_qdist_baseline_negative_sensitivity_FINAL.csv",
)

print("Primary clinical dose rows:", len(clinical_linear))
print("Ordered-factor rows:", len(ordered_factor))
print("Model-interaction rows:", len(model_interactions))
print("Heterogeneity rows:", len(heterogeneity))
print("QDIST restricted rows:", len(qdist_sens))

Primary clinical dose rows: 72
Ordered-factor rows: 216
Model-interaction rows: 48
Heterogeneity rows: 36
QDIST restricted rows: 6


## 4. Exemplar-level sensitivity

The primary Stage-E estimand averages stochastic realizations/exemplars within
participant × transform × dose. Here the individual realization trajectories are
retained and clustered by participant.

For transforms with one exemplar, this sensitivity is algebraically the same
experimental structure as primary. It is most informative for stationary noise,
RIR convolution, and upper-band restriction.

In [6]:
def fit_gee_slope(frame, endpoint):
    d = frame.copy()
    d[endpoint] = pd.to_numeric(d[endpoint], errors="coerce")
    d["dose_code"] = pd.to_numeric(d["dose_code"], errors="coerce")
    d = d.loc[
        np.isfinite(d[endpoint])
        & np.isfinite(d["dose_code"])
        & d["participant_id"].notna()
    ].copy()

    if d["participant_id"].nunique() < 20 or d["dose_code"].nunique() < 2:
        return np.nan, np.nan, np.nan, "insufficient_support"

    try:
        fit = smf.gee(
            f"{endpoint} ~ dose_code",
            groups="participant_id",
            data=d,
            family=sm.families.Gaussian(),
            cov_struct=sm.cov_struct.Exchangeable(),
        ).fit()
        return (
            float(fit.params["dose_code"]),
            float(fit.bse["dose_code"]),
            float(fit.pvalues["dose_code"]),
            "PASS",
        )
    except Exception as exc:
        return np.nan, np.nan, np.nan, f"{type(exc).__name__}: {exc}"

endpoint_map = {
    "diagnosis": ["delta_logit", "delta_error"],
    "severity": ["delta_prediction", "delta_error"],
}

exemplar_rows = []

for task in TASKS:
    task_df = prediction_exemplar.loc[
        prediction_exemplar["task"].eq(task)
    ].copy()

    for transform in sorted(task_df["transform"].dropna().unique()):
        for model in RIDGE_MODELS:
            g = task_df.loc[
                task_df["transform"].eq(transform)
                & task_df["model"].eq(model)
            ].copy()

            for endpoint in endpoint_map[task]:
                local = g[
                    [
                        "participant_id", "transform", "model",
                        "dose_label", "dose_code", "exemplar", endpoint,
                    ]
                ].copy()

                # One zero-dose baseline per observed participant × exemplar trajectory.
                base = (
                    local[["participant_id", "transform", "model", "exemplar"]]
                    .drop_duplicates()
                    .copy()
                )
                base["dose_label"] = "baseline"
                base["dose_code"] = 0
                base[endpoint] = 0.0
                base = base[local.columns]

                analysis = pd.concat([base, local], ignore_index=True)

                est, se, p, status = fit_gee_slope(analysis, endpoint)

                primary = clinical_linear.loc[
                    clinical_linear["task"].eq(task)
                    & clinical_linear["transform"].eq(transform)
                    & clinical_linear["model"].eq(model)
                    & clinical_linear["endpoint"].eq(endpoint)
                ]

                primary_est = (
                    float(primary["estimate"].iloc[0])
                    if len(primary) == 1
                    else np.nan
                )

                exemplar_rows.append({
                    "task": task,
                    "transform": transform,
                    "model": model,
                    "endpoint": endpoint,
                    "participants": analysis["participant_id"].nunique(),
                    "observed_exemplars": int(local["exemplar"].nunique()),
                    "primary_exemplar_averaged_slope": primary_est,
                    "exemplar_level_slope": est,
                    "exemplar_level_se": se,
                    "exemplar_level_p": p,
                    "same_direction_as_primary": (
                        bool(np.sign(est) == np.sign(primary_est))
                        if np.isfinite(est) and np.isfinite(primary_est)
                        and est != 0 and primary_est != 0
                        else np.nan
                    ),
                    "absolute_slope_difference": (
                        float(abs(est - primary_est))
                        if np.isfinite(est) and np.isfinite(primary_est)
                        else np.nan
                    ),
                    "status": status,
                })

exemplar_clinical = pd.DataFrame(exemplar_rows)

if not exemplar_clinical["status"].eq("PASS").all():
    display(exemplar_clinical.loc[~exemplar_clinical["status"].eq("PASS")])
    raise RuntimeError("At least one exemplar-level clinical GEE failed.")

atomic_csv(
    exemplar_clinical,
    TABLES / "goal3_exemplar_level_clinical_sensitivity.csv",
)

# Exemplar-level target-Q behavior.
target_specs = (
    target_q[
        [
            "task", "family", "transform", "target_q",
            "direction_multiplier", "natural_iqr_raw",
        ]
    ]
    .drop_duplicates()
)

target_exemplar_rows = []

for spec in target_specs.to_dict("records"):
    g = feature_exemplar.loc[
        feature_exemplar["task"].eq(spec["task"])
        & feature_exemplar["transform"].eq(spec["transform"])
        & feature_exemplar["measurement_type"].eq("Q")
        & feature_exemplar["feature"].eq(spec["target_q"])
    ].copy()

    if g.empty:
        continue

    scale = float(spec["natural_iqr_raw"])
    g["oriented_delta_iqr"] = (
        float(spec["direction_multiplier"])
        * pd.to_numeric(g["delta"], errors="coerce")
        / scale
    )

    for (dose_label, dose_code, exemplar), z in g.groupby(
        ["dose_label", "dose_code", "exemplar"],
        dropna=False,
    ):
        vals = pd.to_numeric(z["oriented_delta_iqr"], errors="coerce")
        vals = vals[np.isfinite(vals)]
        target_exemplar_rows.append({
            "task": spec["task"],
            "family": spec["family"],
            "transform": spec["transform"],
            "target_q": spec["target_q"],
            "dose_label": dose_label,
            "dose_code": int(dose_code),
            "exemplar": int(exemplar),
            "finite_participants": len(vals),
            "median_oriented_delta_natural_iqr": (
                float(np.median(vals)) if len(vals) else np.nan
            ),
            "expected_direction_fraction": (
                float(np.mean(vals > 0)) if len(vals) else np.nan
            ),
        })

target_exemplar = pd.DataFrame(target_exemplar_rows)
atomic_csv(
    target_exemplar,
    TABLES / "goal3_exemplar_level_targetQ_sensitivity.csv",
)

print("=" * 78)
print("EXEMPLAR-LEVEL SENSITIVITY: PASS")
print("=" * 78)
print("Clinical rows:", len(exemplar_clinical))
print("Target-Q exemplar rows:", len(target_exemplar))

EXEMPLAR-LEVEL SENSITIVITY: PASS
Clinical rows: 72
Target-Q exemplar rows: 84


## 5. Segmentation-mediated sensitivity: feasibility and empirical change audit

The frozen Methods specify a fixed-baseline-segmentation sensitivity **where
technically feasible**.

Stage E retained segment counts and support durations for every measured waveform,
but it did not persist the complete sample-level baseline interval masks required
to re-extract all Q and A measures under truly fixed baseline segmentation.

This notebook therefore does **not** fabricate a pseudo-fixed analysis from counts
or durations. It records that a genuine fixed-mask remeasurement would require a
new waveform extraction experiment, and quantifies the segmentation changes
observed under the prespecified end-to-end re-segmentation analysis.

In [7]:
segment_fields = [
    "raw_speech_interval_count",
    "primary_speech_interval_count",
    "strict_speech_interval_count",
    "strict_internal_nonspeech_interval_count",
    "primary_speech_support_sec",
    "strict_speech_support_sec",
    "strict_internal_nonspeech_support_sec",
]

missing_segment_fields = [c for c in segment_fields if c not in response.columns]
if missing_segment_fields:
    raise RuntimeError(
        f"Stage-E response missing segmentation audit fields: {missing_segment_fields}"
    )

interval_mask_like = [
    c for c in response.columns
    if any(token in c.lower() for token in [
        "interval_start", "interval_stop", "interval_end",
        "sample_mask", "segment_mask", "speech_mask",
    ])
]

baseline_seg = (
    response.loc[
        response["transform"].eq("baseline"),
        ["task", "participant_id", *segment_fields],
    ]
    .drop_duplicates(["task", "participant_id"])
    .set_index(["task", "participant_id"])
)

seg_rows = []

for row in response.loc[~response["transform"].eq("baseline")].to_dict("records"):
    key = (row["task"], str(row["participant_id"]))
    b = baseline_seg.loc[key]

    count_changed = False
    support_abs_changes = []

    for c in segment_fields:
        bv = pd.to_numeric(pd.Series([b[c]]), errors="coerce").iloc[0]
        pv = pd.to_numeric(pd.Series([row[c]]), errors="coerce").iloc[0]

        if c.endswith("_count") and np.isfinite(bv) and np.isfinite(pv):
            count_changed = count_changed or (int(bv) != int(pv))
        elif c.endswith("_sec") and np.isfinite(bv) and np.isfinite(pv):
            support_abs_changes.append(abs(float(pv - bv)))

    seg_rows.append({
        "task": row["task"],
        "participant_id": str(row["participant_id"]),
        "family": row["family"],
        "transform": row["transform"],
        "dose_label": row["dose_label"],
        "dose_code": int(row["dose_code"]),
        "exemplar": int(row["exemplar"]),
        "any_interval_count_change": bool(count_changed),
        "max_abs_support_change_sec": (
            max(support_abs_changes) if support_abs_changes else np.nan
        ),
    })

segmentation_change = pd.DataFrame(seg_rows)

segmentation_summary = (
    segmentation_change.groupby(
        ["task", "family", "transform", "dose_label", "dose_code"],
        as_index=False,
    )
    .agg(
        rows=("participant_id", "size"),
        participants=("participant_id", "nunique"),
        fraction_rows_with_interval_count_change=(
            "any_interval_count_change", "mean"
        ),
        median_max_abs_support_change_sec=(
            "max_abs_support_change_sec", "median"
        ),
        q95_max_abs_support_change_sec=(
            "max_abs_support_change_sec",
            lambda s: float(np.nanquantile(
                pd.to_numeric(s, errors="coerce"), 0.95
            )),
        ),
    )
)

atomic_csv(
    segmentation_change,
    TABLES / "goal3_segmentation_change_exemplar.csv",
)
atomic_csv(
    segmentation_summary,
    TABLES / "goal3_segmentation_change_summary.csv",
)

fixed_segmentation_disposition = {
    "created_utc": utc_now(),
    "status": "NOT_TECHNICALLY_FEASIBLE_FROM_FROZEN_STAGEE_ARTIFACTS",
    "reason": (
        "Stage E retained segmentation counts/support durations but not complete "
        "sample-level baseline interval masks. A genuine fixed-baseline-segmentation "
        "sensitivity would require a new waveform remeasurement pass. Counts/durations "
        "are not substituted for fixed masks."
    ),
    "interval_mask_like_columns_found": interval_mask_like,
    "primary_analysis": "full end-to-end re-segmentation retained",
    "supplementary_segmentation_change_audit": (
        "goal3_segmentation_change_summary.csv"
    ),
}

atomic_json(
    fixed_segmentation_disposition,
    AUDIT / "goal3_fixed_baseline_segmentation_disposition.json",
)

print("FIXED-BASELINE SEGMENTATION DISPOSITION:")
print(fixed_segmentation_disposition["status"])
print("Interval-mask-like persisted columns:", interval_mask_like)
display(segmentation_summary.head(20))

FIXED-BASELINE SEGMENTATION DISPOSITION:
NOT_TECHNICALLY_FEASIBLE_FROM_FROZEN_STAGEE_ARTIFACTS
Interval-mask-like persisted columns: []


,task,family,transform,dose_label,dose_code,rows,participants,fraction_rows_with_interval_count_change,median_max_abs_support_change_sec,q95_max_abs_support_change_sec
0,diagnosis,QADD,stationary_colored_broadband,high,3,995,199,0.833166,0.836,4.3916
1,diagnosis,QADD,stationary_colored_broadband,low,1,995,199,0.562814,0.324,1.6192
2,diagnosis,QADD,stationary_colored_broadband,medium,2,995,199,0.638191,0.384,1.9612
3,diagnosis,QCHAN,upper_band_restriction,high,3,597,199,0.773869,0.452,3.9168
4,diagnosis,QCHAN,upper_band_restriction,low,1,597,199,0.432161,0.100,0.8160
5,diagnosis,QCHAN,upper_band_restriction,medium,2,597,199,0.569514,0.224,2.1984
6,diagnosis,QDIST,symmetric_hard_clipping,high,3,199,199,0.125628,0.032,0.4360
7,diagnosis,QDIST,symmetric_hard_clipping,low,1,199,199,0.025126,0.000,0.0640
8,diagnosis,QDIST,symmetric_hard_clipping,medium,2,199,199,0.060302,0.000,0.2024
9,diagnosis,QGAIN,smooth_time_varying_gain,high,3,199,199,0.909548,0.708,11.2996


## def find_one(patterns):
    hits = []

    for pattern in patterns:
        hits.extend(
            ROOT.rglob(pattern)
        )

    hits = sorted(
        set(hits)
    )

    if not hits:
        raise FileNotFoundError(
            f"Could not find any of: {patterns}"
        )

    # Prefer v1_0_1 when both versions exist.
    hits = sorted(
        hits,
        key=lambda p: (
            "v1_0_1" not in p.name,
            "v1_0" not in p.name,
            len(str(p)),
        ),
    )

    return hits[0]


# =====================================================================
# LOCATE AUTHORITATIVE GOAL-2 -> GOAL-3 BRIDGE NOTEBOOK
# =====================================================================

bridge_notebook = find_one([
    "34_goal3_stage_d_goal2_model_bundle_bridge_final_v1_0_1.ipynb",
    "34_goal3_stage_d_goal2_model_bundle_bridge_final_v1_0.ipynb",
])


print(
    "Using bridge notebook prelude:",
    bridge_notebook,
)


# =====================================================================
# EXECUTE ONLY THE PRE-MODEL-BUILD PORTION OF NOTEBOOK 34
# =====================================================================

def execute_bridge_prelude(path: Path):

    nb = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    ns = {
        "__name__": "__goal3_hgb_bridge_prelude__",
        "__file__": str(path),
    }

    stopped = False

    for idx, cell in enumerate(
        nb["cells"]
    ):

        source = "".join(
            cell.get(
                "source",
                [],
            )
        )

        # -------------------------------------------------------------
        # Stop before Notebook 34 starts creating the five frozen
        # Goal-2 ridge bundle files.
        # -------------------------------------------------------------
        if (
            cell.get("cell_type")
            == "markdown"
        ):

            if (
                "## 7. Build the five outer-fold bundle files"
                in source
            ):
                stopped = True
                break

            continue

        if (
            cell.get("cell_type")
            != "code"
        ):
            continue

        if not source.strip():
            continue

        exec(
            compile(
                source,
                (
                    f"{path.name}:"
                    f"cell_{idx}"
                ),
                "exec",
            ),
            ns,
            ns,
        )

    if not stopped:
        raise RuntimeError(
            "Could not find the Notebook-34 "
            "model-build stop heading."
        )

    return ns


bridge = execute_bridge_prelude(
    bridge_notebook
)


# =====================================================================
# VERIFY REQUIRED NOTEBOOK-34 OBJECTS
# =====================================================================
#
# Notebook 34 calls the acoustic-support mapping A_SUPPORT.
#
# IMPORTANT:
# A_SUPPORT contains the full frozen acoustic-feature registry,
# not only the six Primary-A features.
# =====================================================================

required_bridge_names = [
    "TASK_FRAMES",
    "split_manifest",
    "participant_weights",
    "build_qchan_context",
    "fit_model_state",
    "predict_from_state",
    "PRIMARY_SPECS",
    "PRIMARY_A",
    "CORE_Q",
    "QCHAN",
    "AGE",
    "A_SUPPORT",
]


missing_bridge = [
    name
    for name in required_bridge_names
    if name not in bridge
]


if missing_bridge:
    raise RuntimeError(
        "Notebook-34 prelude missing "
        "required objects: "
        f"{missing_bridge}"
    )


# =====================================================================
# NORMALIZE A-SUPPORT MAPPING FOR NOTEBOOK 36
# =====================================================================
#
# Notebook 34:
#
#     A_SUPPORT
#
# contains mappings for the full acoustic registry.
#
# Notebook 36 only needs support indicators for the frozen Primary-A
# features. Therefore we construct an explicit Primary-A-only alias:
#
#     A_SUPPORT_MAP
#
# This does NOT modify the authoritative Notebook-34 mapping.
# =====================================================================

primary_a = list(
    bridge["PRIMARY_A"]
)

source_support_map = dict(
    bridge["A_SUPPORT"]
)


# ---------------------------------------------------------------------
# Every frozen Primary-A feature must have a support mapping.
# ---------------------------------------------------------------------

missing_primary_support = [
    feature
    for feature in primary_a
    if feature not in source_support_map
]


if missing_primary_support:
    raise RuntimeError(
        "A_SUPPORT is missing support mappings "
        "for frozen Primary-A features: "
        f"{missing_primary_support}"
    )


# ---------------------------------------------------------------------
# Create Primary-A-only mapping for Notebook 36.
# ---------------------------------------------------------------------

bridge["A_SUPPORT_MAP"] = {
    feature: source_support_map[feature]
    for feature in primary_a
}


# ---------------------------------------------------------------------
# Validate mapped support-column names.
# ---------------------------------------------------------------------

invalid_support_columns = {
    feature: support_col
    for feature, support_col
    in bridge["A_SUPPORT_MAP"].items()
    if (
        support_col is None
        or str(support_col).strip() == ""
        or str(support_col).strip().lower()
        in {
            "nan",
            "none",
            "null",
        }
    )
}


if invalid_support_columns:
    raise RuntimeError(
        "Invalid Primary-A support-column mappings: "
        f"{invalid_support_columns}"
    )


# ---------------------------------------------------------------------
# Final Primary-A mapping contract.
# ---------------------------------------------------------------------

if set(
    bridge["A_SUPPORT_MAP"].keys()
) != set(primary_a):
    raise RuntimeError(
        "Primary-A-only support mapping "
        "construction failed."
    )


if len(
    bridge["A_SUPPORT_MAP"]
) != len(primary_a):
    raise RuntimeError(
        "Primary-A support mapping contains "
        "duplicate or inconsistent entries."
    )


print(
    "Bridge acoustic-support mapping:",
    "A_SUPPORT -> Primary-A-only "
    "A_SUPPORT_MAP READY",
)

print(
    "Primary-A support indicators:"
)

for feature in primary_a:
    print(
        "  ",
        feature,
        "->",
        bridge["A_SUPPORT_MAP"][feature],
    )


# =====================================================================
# LOAD ALREADY-FROZEN GOAL-2 HGB FOLD MANIFEST
# =====================================================================

goal2_hgb_manifest = safe_csv(
    GOAL2
    / "tables"
    / "goal2_hgb_fold_manifest.csv",
    required=[
        "task",
        "repeat",
        "outer_fold",
        "model",
        "selected_params",
        "selected_residualizer_alpha",
    ],
)


# =====================================================================
# RESTRICT TO OUTER REPEAT 1
# =====================================================================
#
# Controlled Goal 3 is frozen to the first outer repeat.
#
# Expected:
#
#   2 tasks
# × 5 folds
# × 3 HGB model representations
# = 30 rows
# =====================================================================

repeat1_manifest = (
    goal2_hgb_manifest.loc[
        pd.to_numeric(
            goal2_hgb_manifest[
                "repeat"
            ],
            errors="coerce",
        ).eq(1)
    ]
    .copy()
)


expected_hgb_rows = (
    len(TASKS)
    * OUTER_FOLDS
    * len(RIDGE_MODELS)
)


if len(repeat1_manifest) != expected_hgb_rows:
    raise RuntimeError(
        "Expected "
        f"{expected_hgb_rows} "
        "frozen first-repeat HGB fold rows; "
        f"found {len(repeat1_manifest)}."
    )


# =====================================================================
# VALIDATE TASK CONTRACT
# =====================================================================

observed_tasks = set(
    repeat1_manifest[
        "task"
    ]
    .astype(str)
)


expected_tasks = set(
    TASKS
)


if observed_tasks != expected_tasks:
    raise RuntimeError(
        "Unexpected HGB task set. "
        f"Expected {sorted(expected_tasks)}, "
        f"found {sorted(observed_tasks)}."
    )


# =====================================================================
# VALIDATE OUTER-FOLD CONTRACT
# =====================================================================

observed_folds = set(
    pd.to_numeric(
        repeat1_manifest[
            "outer_fold"
        ],
        errors="raise",
    )
    .astype(int)
)


expected_folds = set(
    range(
        1,
        OUTER_FOLDS + 1,
    )
)


if observed_folds != expected_folds:
    raise RuntimeError(
        "Unexpected first-repeat HGB "
        "outer-fold set. "
        f"Expected {sorted(expected_folds)}, "
        f"found {sorted(observed_folds)}."
    )


# =====================================================================
# VALIDATE THREE HGB REPRESENTATIONS PER TASK × FOLD
# =====================================================================

structure = (
    repeat1_manifest
    .groupby(
        [
            "task",
            "outer_fold",
        ],
        dropna=False,
    )
    .agg(
        rows=("model", "size"),
        unique_models=("model", "nunique"),
    )
    .reset_index()
)


if not structure[
    "rows"
].eq(
    len(RIDGE_MODELS)
).all():

    display(structure)

    raise RuntimeError(
        "At least one task × fold does not "
        "contain exactly three frozen "
        "HGB model rows."
    )


if not structure[
    "unique_models"
].eq(
    len(RIDGE_MODELS)
).all():

    display(structure)

    raise RuntimeError(
        "At least one task × fold does not "
        "contain three unique HGB models."
    )


# =====================================================================
# VALIDATE SELECTED-PARAMETER FIELDS ARE POPULATED
# =====================================================================

if repeat1_manifest[
    "selected_params"
].isna().any():

    bad = repeat1_manifest.loc[
        repeat1_manifest[
            "selected_params"
        ].isna()
    ].copy()

    display(bad)

    raise RuntimeError(
        "At least one frozen HGB fold row "
        "has missing selected_params."
    )


# Confirm selected_params values are valid JSON/dict-like objects.
parsed_parameter_rows = []

for idx, row in repeat1_manifest.iterrows():

    raw = row[
        "selected_params"
    ]

    if isinstance(
        raw,
        dict,
    ):
        params = raw

    else:
        raw_text = str(
            raw
        ).strip()

        try:
            params = json.loads(
                raw_text
            )

        except Exception:
            try:
                params = pyast.literal_eval(
                    raw_text
                )

            except Exception as exc:
                raise RuntimeError(
                    "Could not parse frozen HGB "
                    "selected_params for "
                    f"task={row['task']}, "
                    f"fold={row['outer_fold']}, "
                    f"model={row['model']}: "
                    f"{exc}"
                )

    if not isinstance(
        params,
        dict,
    ):
        raise RuntimeError(
            "Frozen HGB selected_params is "
            "not dictionary-like for "
            f"task={row['task']}, "
            f"fold={row['outer_fold']}, "
            f"model={row['model']}."
        )

    parsed_parameter_rows.append(
        params
    )


repeat1_manifest[
    "_parsed_selected_params"
] = parsed_parameter_rows


# =====================================================================
# FINAL CONTRACT REPORT
# =====================================================================

print(
    "=" * 78
)

print(
    "GOAL-2 HGB INPUT CONTRACT: PASS"
)

print(
    "=" * 78
)

print(
    "Frozen first-repeat HGB manifest rows:",
    len(repeat1_manifest),
)

print(
    "Expected rows:",
    expected_hgb_rows,
)

print(
    "Tasks:",
    sorted(
        observed_tasks
    ),
)

print(
    "Outer folds:",
    sorted(
        observed_folds
    ),
)

print(
    "Primary-A features:",
    len(
        bridge[
            "PRIMARY_A"
        ]
    ),
)

print(
    "Core-Q features:",
    len(
        bridge[
            "CORE_Q"
        ]
    ),
)

print(
    "QCHAN features:",
    len(
        bridge[
            "QCHAN"
        ]
    ),
)

print(
    "Full A_SUPPORT registry entries:",
    len(
        bridge[
            "A_SUPPORT"
        ]
    ),
)

print(
    "Primary-A-only A_SUPPORT_MAP entries:",
    len(
        bridge[
            "A_SUPPORT_MAP"
        ]
    ),
)

print(
    "Task × fold HGB structure rows:",
    len(
        structure
    ),
)

print(
    "All selected HGB parameter objects parse:",
    True,
)6. Reconstruct the frozen first-repeat HGB sensitivity states

This is the only model-fitting stage in Notebook 36.

It is **not model development**:

- the HGB grid was frozen in Goal 2;
- every selected HGB parameter and iteration count is read from the already
  completed Goal-2 fold checkpoints;
- the residualizer penalty is read from those same frozen checkpoints;
- the participant split is fixed to outer repeat 1;
- only unmodified outer-training observations are used;
- no Stage-E outcome can alter any HGB setting.

Before HGB is applied to Goal-3 perturbations, the reconstructed state must
reproduce the original Goal-2 HGB repeat-1 OOF predictions within `1e-10`.

In [11]:
def find_one(patterns):
    hits = []

    for pattern in patterns:
        hits.extend(
            ROOT.rglob(pattern)
        )

    hits = sorted(
        set(hits)
    )

    if not hits:
        raise FileNotFoundError(
            f"Could not find any of: {patterns}"
        )

    # Prefer v1_0_1 when both versions exist.
    hits = sorted(
        hits,
        key=lambda p: (
            "v1_0_1" not in p.name,
            "v1_0" not in p.name,
            len(str(p)),
        ),
    )

    return hits[0]


# =====================================================================
# LOCATE AUTHORITATIVE GOAL-2 -> GOAL-3 BRIDGE NOTEBOOK
# =====================================================================

bridge_notebook = find_one([
    "34_goal3_stage_d_goal2_model_bundle_bridge_final_v1_0_1.ipynb",
    "34_goal3_stage_d_goal2_model_bundle_bridge_final_v1_0.ipynb",
])


print(
    "Using bridge notebook prelude:",
    bridge_notebook,
)


# =====================================================================
# EXECUTE ONLY THE PRE-MODEL-BUILD PORTION OF NOTEBOOK 34
# =====================================================================

def execute_bridge_prelude(path: Path):

    nb = json.loads(
        Path(path).read_text(
            encoding="utf-8"
        )
    )

    ns = {
        "__name__": "__goal3_hgb_bridge_prelude__",
        "__file__": str(path),
    }

    stopped = False

    for idx, cell in enumerate(
        nb["cells"]
    ):

        source = "".join(
            cell.get(
                "source",
                [],
            )
        )

        # -------------------------------------------------------------
        # Stop before Notebook 34 starts creating the five frozen
        # Goal-2 ridge bundle files.
        # -------------------------------------------------------------
        if (
            cell.get("cell_type")
            == "markdown"
        ):

            if (
                "## 7. Build the five outer-fold bundle files"
                in source
            ):
                stopped = True
                break

            continue

        if (
            cell.get("cell_type")
            != "code"
        ):
            continue

        if not source.strip():
            continue

        exec(
            compile(
                source,
                (
                    f"{path.name}:"
                    f"cell_{idx}"
                ),
                "exec",
            ),
            ns,
            ns,
        )

    if not stopped:
        raise RuntimeError(
            "Could not find the Notebook-34 "
            "model-build stop heading."
        )

    return ns


bridge = execute_bridge_prelude(
    bridge_notebook
)


# =====================================================================
# VERIFY REQUIRED NOTEBOOK-34 OBJECTS
# =====================================================================
#
# Notebook 34 calls the acoustic-support mapping A_SUPPORT.
#
# IMPORTANT:
# A_SUPPORT contains the full frozen acoustic-feature registry,
# not only the six Primary-A features.
# =====================================================================

required_bridge_names = [
    "TASK_FRAMES",
    "split_manifest",
    "participant_weights",
    "build_qchan_context",
    "fit_model_state",
    "predict_from_state",
    "PRIMARY_SPECS",
    "PRIMARY_A",
    "CORE_Q",
    "QCHAN",
    "AGE",
    "A_SUPPORT",
]


missing_bridge = [
    name
    for name in required_bridge_names
    if name not in bridge
]


if missing_bridge:
    raise RuntimeError(
        "Notebook-34 prelude missing "
        "required objects: "
        f"{missing_bridge}"
    )


# =====================================================================
# NORMALIZE A-SUPPORT MAPPING FOR NOTEBOOK 36
# =====================================================================
#
# Notebook 34:
#
#     A_SUPPORT
#
# contains mappings for the full acoustic registry.
#
# Notebook 36 only needs support indicators for the frozen Primary-A
# features. Therefore we construct an explicit Primary-A-only alias:
#
#     A_SUPPORT_MAP
#
# This does NOT modify the authoritative Notebook-34 mapping.
# =====================================================================

primary_a = list(
    bridge["PRIMARY_A"]
)

source_support_map = dict(
    bridge["A_SUPPORT"]
)


# ---------------------------------------------------------------------
# Every frozen Primary-A feature must have a support mapping.
# ---------------------------------------------------------------------

missing_primary_support = [
    feature
    for feature in primary_a
    if feature not in source_support_map
]


if missing_primary_support:
    raise RuntimeError(
        "A_SUPPORT is missing support mappings "
        "for frozen Primary-A features: "
        f"{missing_primary_support}"
    )


# ---------------------------------------------------------------------
# Create Primary-A-only mapping for Notebook 36.
# ---------------------------------------------------------------------

bridge["A_SUPPORT_MAP"] = {
    feature: source_support_map[feature]
    for feature in primary_a
}


# ---------------------------------------------------------------------
# Validate mapped support-column names.
# ---------------------------------------------------------------------

invalid_support_columns = {
    feature: support_col
    for feature, support_col
    in bridge["A_SUPPORT_MAP"].items()
    if (
        support_col is None
        or str(support_col).strip() == ""
        or str(support_col).strip().lower()
        in {
            "nan",
            "none",
            "null",
        }
    )
}


if invalid_support_columns:
    raise RuntimeError(
        "Invalid Primary-A support-column mappings: "
        f"{invalid_support_columns}"
    )


# ---------------------------------------------------------------------
# Final Primary-A mapping contract.
# ---------------------------------------------------------------------

if set(
    bridge["A_SUPPORT_MAP"].keys()
) != set(primary_a):
    raise RuntimeError(
        "Primary-A-only support mapping "
        "construction failed."
    )


if len(
    bridge["A_SUPPORT_MAP"]
) != len(primary_a):
    raise RuntimeError(
        "Primary-A support mapping contains "
        "duplicate or inconsistent entries."
    )


print(
    "Bridge acoustic-support mapping:",
    "A_SUPPORT -> Primary-A-only "
    "A_SUPPORT_MAP READY",
)

print(
    "Primary-A support indicators:"
)

for feature in primary_a:
    print(
        "  ",
        feature,
        "->",
        bridge["A_SUPPORT_MAP"][feature],
    )


# =====================================================================
# LOAD ALREADY-FROZEN GOAL-2 HGB FOLD MANIFEST
# =====================================================================

goal2_hgb_manifest = safe_csv(
    GOAL2
    / "tables"
    / "goal2_hgb_fold_manifest.csv",
    required=[
        "task",
        "repeat",
        "outer_fold",
        "model",
        "selected_params",
        "selected_residualizer_alpha",
    ],
)


# =====================================================================
# RESTRICT TO OUTER REPEAT 1
# =====================================================================
#
# Controlled Goal 3 is frozen to the first outer repeat.
#
# Expected:
#
#   2 tasks
# × 5 folds
# × 3 HGB model representations
# = 30 rows
# =====================================================================

repeat1_manifest = (
    goal2_hgb_manifest.loc[
        pd.to_numeric(
            goal2_hgb_manifest[
                "repeat"
            ],
            errors="coerce",
        ).eq(1)
    ]
    .copy()
)


expected_hgb_rows = (
    len(TASKS)
    * OUTER_FOLDS
    * len(RIDGE_MODELS)
)


if len(repeat1_manifest) != expected_hgb_rows:
    raise RuntimeError(
        "Expected "
        f"{expected_hgb_rows} "
        "frozen first-repeat HGB fold rows; "
        f"found {len(repeat1_manifest)}."
    )


# =====================================================================
# VALIDATE TASK CONTRACT
# =====================================================================

observed_tasks = set(
    repeat1_manifest[
        "task"
    ]
    .astype(str)
)


expected_tasks = set(
    TASKS
)


if observed_tasks != expected_tasks:
    raise RuntimeError(
        "Unexpected HGB task set. "
        f"Expected {sorted(expected_tasks)}, "
        f"found {sorted(observed_tasks)}."
    )


# =====================================================================
# VALIDATE OUTER-FOLD CONTRACT
# =====================================================================

observed_folds = set(
    pd.to_numeric(
        repeat1_manifest[
            "outer_fold"
        ],
        errors="raise",
    )
    .astype(int)
)


expected_folds = set(
    range(
        1,
        OUTER_FOLDS + 1,
    )
)


if observed_folds != expected_folds:
    raise RuntimeError(
        "Unexpected first-repeat HGB "
        "outer-fold set. "
        f"Expected {sorted(expected_folds)}, "
        f"found {sorted(observed_folds)}."
    )


# =====================================================================
# VALIDATE THREE HGB REPRESENTATIONS PER TASK × FOLD
# =====================================================================

structure = (
    repeat1_manifest
    .groupby(
        [
            "task",
            "outer_fold",
        ],
        dropna=False,
    )
    .agg(
        rows=("model", "size"),
        unique_models=("model", "nunique"),
    )
    .reset_index()
)


if not structure[
    "rows"
].eq(
    len(RIDGE_MODELS)
).all():

    display(structure)

    raise RuntimeError(
        "At least one task × fold does not "
        "contain exactly three frozen "
        "HGB model rows."
    )


if not structure[
    "unique_models"
].eq(
    len(RIDGE_MODELS)
).all():

    display(structure)

    raise RuntimeError(
        "At least one task × fold does not "
        "contain three unique HGB models."
    )


# =====================================================================
# VALIDATE SELECTED-PARAMETER FIELDS ARE POPULATED
# =====================================================================

if repeat1_manifest[
    "selected_params"
].isna().any():

    bad = repeat1_manifest.loc[
        repeat1_manifest[
            "selected_params"
        ].isna()
    ].copy()

    display(bad)

    raise RuntimeError(
        "At least one frozen HGB fold row "
        "has missing selected_params."
    )


# Confirm selected_params values are valid JSON/dict-like objects.
parsed_parameter_rows = []

for idx, row in repeat1_manifest.iterrows():

    raw = row[
        "selected_params"
    ]

    if isinstance(
        raw,
        dict,
    ):
        params = raw

    else:
        raw_text = str(
            raw
        ).strip()

        try:
            params = json.loads(
                raw_text
            )

        except Exception:
            try:
                params = pyast.literal_eval(
                    raw_text
                )

            except Exception as exc:
                raise RuntimeError(
                    "Could not parse frozen HGB "
                    "selected_params for "
                    f"task={row['task']}, "
                    f"fold={row['outer_fold']}, "
                    f"model={row['model']}: "
                    f"{exc}"
                )

    if not isinstance(
        params,
        dict,
    ):
        raise RuntimeError(
            "Frozen HGB selected_params is "
            "not dictionary-like for "
            f"task={row['task']}, "
            f"fold={row['outer_fold']}, "
            f"model={row['model']}."
        )

    parsed_parameter_rows.append(
        params
    )


repeat1_manifest[
    "_parsed_selected_params"
] = parsed_parameter_rows


# =====================================================================
# FINAL CONTRACT REPORT
# =====================================================================

print(
    "=" * 78
)

print(
    "GOAL-2 HGB INPUT CONTRACT: PASS"
)

print(
    "=" * 78
)

print(
    "Frozen first-repeat HGB manifest rows:",
    len(repeat1_manifest),
)

print(
    "Expected rows:",
    expected_hgb_rows,
)

print(
    "Tasks:",
    sorted(
        observed_tasks
    ),
)

print(
    "Outer folds:",
    sorted(
        observed_folds
    ),
)

print(
    "Primary-A features:",
    len(
        bridge[
            "PRIMARY_A"
        ]
    ),
)

print(
    "Core-Q features:",
    len(
        bridge[
            "CORE_Q"
        ]
    ),
)

print(
    "QCHAN features:",
    len(
        bridge[
            "QCHAN"
        ]
    ),
)

print(
    "Full A_SUPPORT registry entries:",
    len(
        bridge[
            "A_SUPPORT"
        ]
    ),
)

print(
    "Primary-A-only A_SUPPORT_MAP entries:",
    len(
        bridge[
            "A_SUPPORT_MAP"
        ]
    ),
)

print(
    "Task × fold HGB structure rows:",
    len(
        structure
    ),
)

print(
    "All selected HGB parameter objects parse:",
    True,
)

Using bridge notebook prelude: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\notebooks\34_goal3_Goal2_model_bundle_bridge_FINAL_v1_0_1.ipynb
Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal3-goal2-model-bundle-bridge-v1.0.0
Outer repeat frozen for controlled Goal 3: 1
Prediction reproduction tolerance: 1e-10 1e-10
GOAL 3 BRIDGE PREREQUISITE GATE: PASS
Run signature: 13154dc87de7fe2e
Stage-B transforms: ['RIR_convolution_RMS_matched', 'smooth_time_varying_gain', 'stationary_colored_broadband', 'symmetric_hard_clipping', 'uniform_level_shift', 'upper_band_restriction']


,family,transform,dose_label,candidate_value,candidate_unit
2,QADD,stationary_colored_broadband,high,10.000,dB injected SNR
0,QADD,stationary_colored_broadband,low,35.000,dB injected SNR
1,QADD,stationary_colored_broadband,medium,30.000,dB injected SNR
5,QCHAN,upper_band_restriction,high,2000.000,low-pass cutoff Hz
3,QCHAN,upper_band_restriction,low,4500.000,low-pass cutoff Hz
4,QCHAN,upper_band_restriction,medium,3500.000,low-pass cutoff Hz
8,QDIST,symmetric_hard_clipping,high,0.010,target changed channel-sample fraction
6,QDIST,symmetric_hard_clipping,low,0.001,target changed channel-sample fraction
7,QDIST,symmetric_hard_clipping,medium,0.003,target changed channel-sample fraction
11,QGAIN,smooth_time_varying_gain,high,24.000,dB modulation amplitude


AUTHORITATIVE INPUT / FEATURE CONTRACT GATE: PASS
Primary-A: ['bamboo_percent_pause_time_300ms', 'bamboo_pause_mean_sec_300ms', 'bamboo_phrase_mean_sec_300ms', 'bamboo_phrase_cv_300ms', 'bamboo_nominal_articulation_rate_syll_per_sec', 'bamboo_f0_iqr_semitones']
Core-Q: ['qadd_pause_ac_level_dbfs_median', 'qadd_pause_level_iqr_db', 'qadd_speech_pause_level_contrast_db', 'qgain_typical_speech_level_dbfs', 'qgain_within_segment_iqr_db', 'qgain_between_segment_mad_db', 'qgain_abs_drift_db_per_min', 'qrev_srmr_norm', 'qchan_ltas_distance_db', 'qchan_rolloff95_deficit_hz', 'qchan_highband_ratio_deficit', 'qchan_tilt_steepening_db_per_oct']
Paper 1 commit: cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8


,task,rows,participants,outcome
0,diagnosis,483,199,ALS vs control
1,severity,398,145,ALSFRS-R bulbar subscore


EXACT GOAL 2 POPULATION RECONSTRUCTION: PASS
TASK/FOLD-SPECIFIC QCHAN BUILDER: READY
EXACT PREPROCESSING / RESIDUALIZER STATE FUNCTIONS: READY
FROZEN MODEL-STATE FIT / INFERENCE INTERFACE: READY
Bridge acoustic-support mapping: A_SUPPORT -> Primary-A-only A_SUPPORT_MAP READY
Primary-A support indicators:
   bamboo_percent_pause_time_300ms -> bamboo_percent_pause_time_300ms__supported
   bamboo_pause_mean_sec_300ms -> bamboo_pause_mean_sec_300ms__supported
   bamboo_phrase_mean_sec_300ms -> bamboo_phrase_mean_sec_300ms__supported
   bamboo_phrase_cv_300ms -> bamboo_phrase_cv_300ms__supported
   bamboo_nominal_articulation_rate_syll_per_sec -> bamboo_nominal_articulation_rate_syll_per_sec__supported
   bamboo_f0_iqr_semitones -> bamboo_f0_iqr_semitones__supported
GOAL-2 HGB INPUT CONTRACT: PASS
Frozen first-repeat HGB manifest rows: 30
Expected rows: 30
Tasks: ['diagnosis', 'severity']
Outer folds: [1, 2, 3, 4, 5]
Primary-A features: 6
Core-Q features: 12
QCHAN features: 4
Full A_SUPPORT

In [13]:
import re


# =====================================================================
# HGB CONSTRUCTION
# =====================================================================

def make_hgb(task, params, seed):

    required = [
        "max_depth",
        "learning_rate",
        "min_samples_leaf",
        "l2_regularization",
        "max_iter",
    ]

    missing = [
        key
        for key in required
        if key not in params
    ]

    if missing:
        raise RuntimeError(
            "Frozen HGB parameter object is missing: "
            f"{missing}"
        )

    common = dict(
        max_depth=int(
            params["max_depth"]
        ),
        learning_rate=float(
            params["learning_rate"]
        ),
        min_samples_leaf=int(
            params["min_samples_leaf"]
        ),
        l2_regularization=float(
            params["l2_regularization"]
        ),
        max_iter=int(
            params["max_iter"]
        ),
        early_stopping=False,
        random_state=int(
            seed % (2**31 - 1)
        ),
    )

    if task == "diagnosis":
        return HistGradientBoostingClassifier(
            **common
        )

    if task == "severity":
        return HistGradientBoostingRegressor(
            **common
        )

    raise ValueError(
        f"Unknown task: {task!r}"
    )


# =====================================================================
# EXACT GOAL-2 CHECKPOINT SLUGGING
# =====================================================================
#
# Notebook 21 used:
#
#     out_dir = CHECKPOINTS / _slug(label) / task
#
# with:
#
#     _slug(text) =
#         re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")
#
# Therefore:
#
#     HGB_M_A       -> HGB_M_A
#     HGB_M_A+Q     -> HGB_M_A_Q
#     HGB_M_A-resQ  -> HGB_M_A-resQ
#
# The MODEL LABEL itself is NOT changed.
# Only the checkpoint-directory name is slugged.
# =====================================================================

def goal2_checkpoint_slug(text):

    return re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(text),
    ).strip("_")


# =====================================================================
# PARSE FROZEN PARAMETER OBJECT
# =====================================================================

def parse_selected_params(raw):

    if isinstance(raw, dict):
        params = dict(raw)

    else:
        text = str(raw).strip()

        try:
            params = json.loads(
                text
            )

        except Exception:

            try:
                params = pyast.literal_eval(
                    text
                )

            except Exception as exc:
                raise RuntimeError(
                    "Could not parse frozen HGB "
                    f"selected_params: {exc}"
                )

    if not isinstance(
        params,
        dict,
    ):
        raise RuntimeError(
            "Frozen HGB selected_params "
            "is not dictionary-like."
        )

    return params


# =====================================================================
# LOOK UP ONE FROZEN GOAL-2 HGB MANIFEST ROW
# =====================================================================

def lookup_hgb_manifest_row(
    task,
    model_name,
    fold,
):

    label = f"HGB_{model_name}"

    rows = repeat1_manifest.loc[
        repeat1_manifest[
            "task"
        ].astype(str).eq(
            str(task)
        )
        & pd.to_numeric(
            repeat1_manifest[
                "outer_fold"
            ],
            errors="coerce",
        ).eq(
            int(fold)
        )
        & repeat1_manifest[
            "model"
        ].astype(str).eq(
            label
        )
    ].copy()

    if len(rows) != 1:

        print(
            "\nAVAILABLE MATCHING HGB MANIFEST ROWS"
        )

        display(
            repeat1_manifest.loc[
                repeat1_manifest[
                    "task"
                ].astype(str).eq(
                    str(task)
                )
                & pd.to_numeric(
                    repeat1_manifest[
                        "outer_fold"
                    ],
                    errors="coerce",
                ).eq(
                    int(fold)
                )
            ][
                [
                    "task",
                    "repeat",
                    "outer_fold",
                    "model",
                    "selected_params",
                    "selected_residualizer_alpha",
                ]
            ]
        )

        raise RuntimeError(
            "Expected exactly one frozen "
            "Goal-2 HGB manifest row for "
            f"{task}/fold{fold}/{label}; "
            f"found {len(rows)}."
        )

    return rows.iloc[0]


# =====================================================================
# RESOLVE ACTUAL GOAL-2 HGB CHECKPOINT PATHS
# =====================================================================

def resolve_hgb_checkpoint_paths(
    task,
    model_name,
    fold,
):

    label = f"HGB_{model_name}"

    slug = goal2_checkpoint_slug(
        label
    )

    checkpoint_dir = (
        GOAL2
        / "checkpoints"
        / slug
        / str(task)
    )

    meta_path = (
        checkpoint_dir
        / f"r01_f{int(fold):02d}.json"
    )

    oof_path = (
        checkpoint_dir
        / f"r01_f{int(fold):02d}.csv"
    )

    if not meta_path.exists():

        raise FileNotFoundError(
            "Frozen Goal-2 HGB metadata checkpoint "
            "was not found.\n"
            f"Model label: {label}\n"
            f"Slugged directory: {slug}\n"
            f"Expected path: {meta_path}"
        )

    if not oof_path.exists():

        raise FileNotFoundError(
            "Frozen Goal-2 HGB OOF checkpoint "
            "was not found.\n"
            f"Model label: {label}\n"
            f"Slugged directory: {slug}\n"
            f"Expected path: {oof_path}"
        )

    return (
        label,
        slug,
        meta_path,
        oof_path,
    )


# =====================================================================
# LOAD / CROSS-CHECK ONE FROZEN HGB CHECKPOINT
# =====================================================================

def load_hgb_checkpoint(
    task,
    model_name,
    fold,
):

    manifest_row = (
        lookup_hgb_manifest_row(
            task,
            model_name,
            fold,
        )
    )

    (
        label,
        slug,
        meta_path,
        oof_path,
    ) = resolve_hgb_checkpoint_paths(
        task,
        model_name,
        fold,
    )

    meta = json.loads(
        meta_path.read_text(
            encoding="utf-8"
        )
    )

    authoritative = safe_csv(
        oof_path,
        required=[
            "participant_id",
            "logical_recording_id",
            "repeat",
            "outer_fold",
            "model",
            "y",
            "prediction",
        ],
    )


    # -------------------------------------------------------------
    # Checkpoint identity gates
    # -------------------------------------------------------------

    if str(
        meta.get(
            "task",
            "",
        )
    ) != str(task):

        raise RuntimeError(
            f"{meta_path}: task mismatch."
        )


    if int(
        meta.get(
            "repeat",
            -1,
        )
    ) != 1:

        raise RuntimeError(
            f"{meta_path}: checkpoint is not repeat 1."
        )


    if int(
        meta.get(
            "outer_fold",
            -1,
        )
    ) != int(fold):

        raise RuntimeError(
            f"{meta_path}: outer-fold mismatch."
        )


    if str(
        meta.get(
            "model",
            "",
        )
    ) != label:

        raise RuntimeError(
            f"{meta_path}: model-label mismatch. "
            f"Expected {label!r}; "
            f"found {meta.get('model')!r}."
        )


    # -------------------------------------------------------------
    # Use the sealed Goal-2 manifest as the authoritative source
    # of selected settings.
    # -------------------------------------------------------------

    params = parse_selected_params(
        manifest_row[
            "selected_params"
        ]
    )


    required_params = {
        "max_depth",
        "learning_rate",
        "min_samples_leaf",
        "l2_regularization",
        "max_iter",
    }

    missing_params = (
        required_params
        - set(params)
    )

    if missing_params:

        raise RuntimeError(
            f"{label}/{task}/fold{fold}: "
            "selected HGB parameter object "
            f"is missing {sorted(missing_params)}."
        )


    residualizer_alpha = (
        manifest_row[
            "selected_residualizer_alpha"
        ]
    )

    if pd.isna(
        residualizer_alpha
    ):
        residualizer_alpha = None

    else:
        residualizer_alpha = float(
            residualizer_alpha
        )


    # -------------------------------------------------------------
    # Cross-check metadata selected_params against the sealed table.
    # -------------------------------------------------------------

    meta_params = parse_selected_params(
        meta.get(
            "selected_params",
            {},
        )
    )

    for key in required_params:

        a = float(
            params[key]
        )

        b = float(
            meta_params[key]
        )

        if not np.isclose(
            a,
            b,
            atol=0,
            rtol=0,
        ):

            raise RuntimeError(
                f"{label}/{task}/fold{fold}: "
                f"manifest/checkpoint mismatch "
                f"for {key}: {a} vs {b}."
            )


    # -------------------------------------------------------------
    # OOF identity gates
    # -------------------------------------------------------------

    if not (
        pd.to_numeric(
            authoritative[
                "repeat"
            ],
            errors="raise",
        )
        .astype(int)
        .eq(1)
        .all()
    ):
        raise RuntimeError(
            f"{oof_path}: contains rows "
            "outside repeat 1."
        )


    if not (
        pd.to_numeric(
            authoritative[
                "outer_fold"
            ],
            errors="raise",
        )
        .astype(int)
        .eq(
            int(fold)
        )
        .all()
    ):
        raise RuntimeError(
            f"{oof_path}: contains rows "
            "from another outer fold."
        )


    if not (
        authoritative[
            "model"
        ]
        .astype(str)
        .eq(
            label
        )
        .all()
    ):
        raise RuntimeError(
            f"{oof_path}: model label changed."
        )


    return {
        "label": label,
        "slug": slug,
        "meta_path": meta_path,
        "oof_path": oof_path,
        "meta": meta,
        "params": params,
        "residualizer_alpha": (
            residualizer_alpha
        ),
        "authoritative_oof": (
            authoritative
        ),
    }


# =====================================================================
# MONKEYPATCH ONLY THE FINAL OUTCOME ESTIMATOR
# =====================================================================
#
# fit_model_state() remains the authoritative Notebook-34 preprocessing /
# QCHAN / residualization / final-scaling implementation.
#
# Only its final clinical estimator is replaced with the ALREADY-SELECTED
# Goal-2 HGB estimator.
# =====================================================================

_HGB_CONTEXT = {}


def hgb_fit_outcome_model(
    X,
    y,
    w,
    task,
    ignored_hyperparameter,
):

    params = (
        _HGB_CONTEXT[
            "params"
        ]
    )

    seed = int(
        _HGB_CONTEXT[
            "seed"
        ]
    )

    model = make_hgb(
        task,
        params,
        seed,
    )

    model.fit(
        X,
        np.asarray(
            y,
            dtype=(
                int
                if task == "diagnosis"
                else float
            ),
        ),
        sample_weight=np.asarray(
            w,
            dtype=float,
        ),
    )

    return model


original_fit_outcome_model = (
    bridge[
        "fit_outcome_model"
    ]
)

bridge[
    "fit_outcome_model"
] = hgb_fit_outcome_model


# =====================================================================
# RECONSTRUCT 30 FIRST-REPEAT FROZEN HGB STATES
# =====================================================================

hgb_states = {}

reproduction_rows = []


try:

    for fold in range(
        1,
        OUTER_FOLDS + 1,
    ):

        for (
            task,
            frame,
        ) in bridge[
            "TASK_FRAMES"
        ].items():

            split_manifest = (
                bridge[
                    "split_manifest"
                ]
            )


            heldout_ids = set(
                split_manifest.loc[
                    split_manifest[
                        "repeat"
                    ]
                    .astype(int)
                    .eq(
                        OUTER_REPEAT
                    )
                    & split_manifest[
                        "outer_fold"
                    ]
                    .astype(int)
                    .eq(
                        fold
                    ),
                    "participant_id",
                ]
                .astype(str)
            )


            heldout_ids &= set(
                frame[
                    "participant_id"
                ]
                .astype(str)
            )


            train = frame.loc[
                ~frame[
                    "participant_id"
                ]
                .astype(str)
                .isin(
                    heldout_ids
                )
            ].copy()


            target = frame.loc[
                frame[
                    "participant_id"
                ]
                .astype(str)
                .isin(
                    heldout_ids
                )
            ].copy()


            if (
                train.empty
                or target.empty
            ):
                raise RuntimeError(
                    f"{task}/fold{fold}: "
                    "empty HGB train/test split."
                )


            overlap = (
                set(
                    train[
                        "participant_id"
                    ].astype(str)
                )
                & set(
                    target[
                        "participant_id"
                    ].astype(str)
                )
            )

            if overlap:
                raise RuntimeError(
                    f"{task}/fold{fold}: "
                    "participant leakage detected."
                )


            train[
                "row_weight"
            ] = bridge[
                "participant_weights"
            ](
                train
            )


            target[
                "row_weight"
            ] = bridge[
                "participant_weights"
            ](
                target
            )


            (
                q_train,
                q_target,
                qchan_audit,
            ) = bridge[
                "build_qchan_context"
            ](
                set(
                    train[
                        "participant_id"
                    ].astype(str)
                ),
                train,
                target,
                task,
                fold,
            )


            for model_name in RIDGE_MODELS:

                checkpoint = (
                    load_hgb_checkpoint(
                        task,
                        model_name,
                        fold,
                    )
                )


                label = checkpoint[
                    "label"
                ]

                params = dict(
                    checkpoint[
                        "params"
                    ]
                )

                residualizer_alpha = (
                    checkpoint[
                        "residualizer_alpha"
                    ]
                )

                authoritative = (
                    checkpoint[
                        "authoritative_oof"
                    ].copy()
                )


                # -------------------------------------------------
                # EXACT ORIGINAL GOAL-2 HGB RANDOM SEED
                #
                # Notebook 21 used:
                #
                # seed = deterministic_seed(
                #     "goal2_hgb",
                #     label,
                #     task,
                #     repeat,
                #     outer_fold,
                # )
                #
                # and final HGB fit used seed + 19.
                # -------------------------------------------------

                seed = deterministic_seed(
                    "goal2_hgb",
                    label,
                    task,
                    1,
                    fold,
                )


                _HGB_CONTEXT[
                    "params"
                ] = params

                _HGB_CONTEXT[
                    "seed"
                ] = int(
                    seed + 19
                )


                # -------------------------------------------------
                # FIT EXACT PREPROCESSING + RESIDUALIZATION STATE,
                # WITH FINAL OUTCOME MODEL SWAPPED TO FROZEN HGB
                # -------------------------------------------------

                (
                    state,
                    pred,
                ) = bridge[
                    "fit_model_state"
                ](
                    train,
                    target,
                    task=task,
                    spec=bridge[
                        "PRIMARY_SPECS"
                    ][
                        model_name
                    ],
                    outcome_hyperparameter=1.0,
                    # ignored by HGB monkeypatch
                    residualizer_alpha=(
                        residualizer_alpha
                    ),
                    q_train=q_train,
                    q_target=q_target,
                    qchan_reference_audit=(
                        qchan_audit
                    ),
                )


                state[
                    "outer_fold"
                ] = int(
                    fold
                )

                state[
                    "model"
                ] = label

                state[
                    "selected_hgb_params"
                ] = params

                state[
                    "selected_residualizer_alpha"
                ] = residualizer_alpha

                state[
                    "hgb_seed"
                ] = int(
                    seed + 19
                )

                state[
                    "goal2_hgb_checkpoint_meta"
                ] = str(
                    checkpoint[
                        "meta_path"
                    ]
                )

                state[
                    "goal2_hgb_checkpoint_oof"
                ] = str(
                    checkpoint[
                        "oof_path"
                    ]
                )

                state[
                    "goal2_hgb_checkpoint_slug"
                ] = checkpoint[
                    "slug"
                ]


                # =================================================================
                # EXACT OOF REPRODUCTION
                # =================================================================

                keys = [
                    "participant_id",
                    "logical_recording_id",
                ]


                if (
                    task == "severity"
                    and "assessment_date"
                    in target.columns
                    and "assessment_date"
                    in authoritative.columns
                ):
                    keys.append(
                        "assessment_date"
                    )


                # Normalize identity columns to strings.
                for key in keys:

                    target[
                        key
                    ] = target[
                        key
                    ].astype(str)

                    authoritative[
                        key
                    ] = authoritative[
                        key
                    ].astype(str)


                local = target[
                    keys
                ].copy()


                local[
                    "reconstructed_prediction"
                ] = np.asarray(
                    pred,
                    dtype=float,
                )


                auth = authoritative[
                    keys
                    + [
                        "prediction"
                    ]
                ].copy().rename(
                    columns={
                        "prediction":
                            "authoritative_prediction"
                    }
                )


                compare = local.merge(
                    auth,
                    on=keys,
                    how="inner",
                    validate="one_to_one",
                )


                if (
                    len(compare)
                    != len(local)
                    or len(compare)
                    != len(auth)
                ):

                    print(
                        "\nOOF IDENTITY DIAGNOSTIC"
                    )

                    print(
                        "task:",
                        task,
                    )

                    print(
                        "fold:",
                        fold,
                    )

                    print(
                        "model:",
                        label,
                    )

                    print(
                        "reconstructed rows:",
                        len(local),
                    )

                    print(
                        "authoritative rows:",
                        len(auth),
                    )

                    print(
                        "matched rows:",
                        len(compare),
                    )

                    raise RuntimeError(
                        f"{task}/fold{fold}/{label}: "
                        "HGB OOF identity mismatch."
                    )


                reconstructed_values = (
                    compare[
                        "reconstructed_prediction"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                authoritative_values = (
                    compare[
                        "authoritative_prediction"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )


                diff = np.abs(
                    reconstructed_values
                    - authoritative_values
                )


                max_diff = (
                    float(
                        np.max(
                            diff
                        )
                    )
                    if len(diff)
                    else np.nan
                )


                if not np.allclose(
                    reconstructed_values,
                    authoritative_values,
                    atol=1e-10,
                    rtol=1e-10,
                ):

                    bad_idx = int(
                        np.argmax(
                            diff
                        )
                    )

                    print(
                        "\nHGB REPRODUCTION FAILURE"
                    )

                    print(
                        "Task:",
                        task,
                    )

                    print(
                        "Fold:",
                        fold,
                    )

                    print(
                        "Model:",
                        label,
                    )

                    print(
                        "Checkpoint directory:",
                        checkpoint[
                            "slug"
                        ],
                    )

                    print(
                        "Maximum |difference|:",
                        max_diff,
                    )

                    print(
                        "Worst reconstructed prediction:",
                        reconstructed_values[
                            bad_idx
                        ],
                    )

                    print(
                        "Worst authoritative prediction:",
                        authoritative_values[
                            bad_idx
                        ],
                    )

                    raise RuntimeError(
                        f"{task}/fold{fold}/{label}: "
                        "reconstructed HGB state does "
                        "not reproduce Goal-2 OOF. "
                        f"max |Δ|={max_diff:.3e}"
                    )


                # -------------------------------------------------
                # STORE VERIFIED STATE
                # -------------------------------------------------

                hgb_states[
                    (
                        task,
                        fold,
                        model_name,
                    )
                ] = state


                reproduction_rows.append({
                    "task": task,
                    "outer_repeat": 1,
                    "outer_fold": int(
                        fold
                    ),
                    "model": label,
                    "checkpoint_slug": (
                        checkpoint[
                            "slug"
                        ]
                    ),
                    "rows": int(
                        len(compare)
                    ),
                    "max_abs_prediction_difference": (
                        max_diff
                    ),
                    "selected_params": json.dumps(
                        params,
                        sort_keys=True,
                    ),
                    "selected_residualizer_alpha": (
                        residualizer_alpha
                    ),
                    "checkpoint_meta_path": str(
                        checkpoint[
                            "meta_path"
                        ]
                    ),
                    "checkpoint_oof_path": str(
                        checkpoint[
                            "oof_path"
                        ]
                    ),
                    "status": "PASS",
                })


                print(
                    f"PASS | {task:9s} | "
                    f"fold {fold} | "
                    f"{label:14s} | "
                    f"max |Δ|={max_diff:.3e}"
                )


finally:

    # ALWAYS restore the original ridge outcome fitter,
    # even if a reconstruction gate raises.
    bridge[
        "fit_outcome_model"
    ] = original_fit_outcome_model


# =====================================================================
# FINAL REPRODUCTION TABLE
# =====================================================================

hgb_reproduction = pd.DataFrame(
    reproduction_rows
)


expected_states = (
    len(TASKS)
    * OUTER_FOLDS
    * len(RIDGE_MODELS)
)


if len(
    hgb_reproduction
) != expected_states:

    display(
        hgb_reproduction
    )

    raise RuntimeError(
        "Expected "
        f"{expected_states} "
        "HGB reproduction rows; "
        f"found {len(hgb_reproduction)}."
    )


if not hgb_reproduction[
    "status"
].eq(
    "PASS"
).all():

    raise RuntimeError(
        "At least one HGB state failed "
        "Goal-2 reproduction."
    )


if hgb_reproduction[
    "max_abs_prediction_difference"
].max() > 1e-10:

    raise RuntimeError(
        "At least one HGB reproduction "
        "difference exceeds 1e-10."
    )


atomic_csv(
    hgb_reproduction,
    TABLES
    / "goal3_hgb_goal2_reproduction_audit.csv",
)


# =====================================================================
# PERSIST VERIFIED FIRST-REPEAT HGB STATES
# =====================================================================

hgb_state_path = (
    HGB_DIR
    / "goal3_first_repeat_frozen_HGB_states.joblib"
)


joblib.dump(
    hgb_states,
    hgb_state_path,
    compress=3,
)


if (
    not hgb_state_path.exists()
    or hgb_state_path.stat().st_size == 0
):
    raise RuntimeError(
        "HGB state serialization failed."
    )


# =====================================================================
# FINAL REPORT
# =====================================================================

print()

print(
    "=" * 78
)

print(
    "FROZEN HGB STATE RECONSTRUCTION: PASS"
)

print(
    "=" * 78
)

print(
    "States:",
    len(
        hgb_states
    ),
)

print(
    "Reproduction rows:",
    len(
        hgb_reproduction
    ),
)

print(
    "Maximum absolute Goal-2 HGB "
    "OOF reproduction difference:",
    (
        f"{hgb_reproduction['max_abs_prediction_difference'].max():.3e}"
    ),
)

print(
    "State file:",
    hgb_state_path,
)

print()

print(
    "Checkpoint directories used:"
)

for slug in sorted(
    hgb_reproduction[
        "checkpoint_slug"
    ].unique()
):
    print(
        "  ",
        slug,
    )

PASS | diagnosis | fold 1 | HGB_M_A        | max |Δ|=1.110e-16
PASS | diagnosis | fold 1 | HGB_M_A+Q      | max |Δ|=1.110e-16
PASS | diagnosis | fold 1 | HGB_M_A-resQ   | max |Δ|=1.110e-16
PASS | severity  | fold 1 | HGB_M_A        | max |Δ|=1.776e-15
PASS | severity  | fold 1 | HGB_M_A+Q      | max |Δ|=1.776e-15
PASS | severity  | fold 1 | HGB_M_A-resQ   | max |Δ|=1.776e-15
PASS | diagnosis | fold 2 | HGB_M_A        | max |Δ|=1.110e-16
PASS | diagnosis | fold 2 | HGB_M_A+Q      | max |Δ|=1.110e-16
PASS | diagnosis | fold 2 | HGB_M_A-resQ   | max |Δ|=1.110e-16
PASS | severity  | fold 2 | HGB_M_A        | max |Δ|=1.776e-15
PASS | severity  | fold 2 | HGB_M_A+Q      | max |Δ|=1.776e-15
PASS | severity  | fold 2 | HGB_M_A-resQ   | max |Δ|=1.776e-15
PASS | diagnosis | fold 3 | HGB_M_A        | max |Δ|=1.110e-16
PASS | diagnosis | fold 3 | HGB_M_A+Q      | max |Δ|=1.110e-16
PASS | diagnosis | fold 3 | HGB_M_A-resQ   | max |Δ|=1.110e-16
PASS | severity  | fold 3 | HGB_M_A        | max |Δ|=1.

## 7. Apply the verified frozen HGB states to the already-measured Stage-E rows

No waveform is decoded here. The saved Stage-E Q/A measurements are transformed
through the verified HGB states exactly as the ridge states were used during
Stage E.

In [14]:
PRIMARY_A = list(bridge["PRIMARY_A"])
CORE_Q = list(bridge["CORE_Q"])
QCHAN = list(bridge["QCHAN"])
AGE = bridge["AGE"]
A_SUPPORT_MAP = dict(bridge["A_SUPPORT_MAP"])

def response_row_to_inference_frames(row):
    payload = {
        "participant_id": str(row["participant_id"]),
        "logical_recording_id": str(row["logical_recording_id"]),
        AGE: float(row["age_at_recording_years"]),
    }

    for feature in PRIMARY_A:
        payload[feature] = pd.to_numeric(
            pd.Series([row.get(f"A__{feature}", np.nan)]),
            errors="coerce",
        ).iloc[0]
        support_col = A_SUPPORT_MAP[feature]
        payload[support_col] = pd.to_numeric(
            pd.Series([row.get(f"A_support__{feature}", 0)]),
            errors="coerce",
        ).fillna(0).iloc[0]

    for feature in CORE_Q:
        payload[feature] = pd.to_numeric(
            pd.Series([row.get(feature, np.nan)]),
            errors="coerce",
        ).iloc[0]

    target = pd.DataFrame([payload])

    q_target = pd.DataFrame([{
        "participant_id": str(row["participant_id"]),
        "logical_recording_id": str(row["logical_recording_id"]),
        **{
            feature: pd.to_numeric(
                pd.Series([row.get(feature, np.nan)]),
                errors="coerce",
            ).iloc[0]
            for feature in QCHAN
        },
    }])

    return target, q_target

hgb_response_rows = []

for i, row in enumerate(response.to_dict("records"), start=1):
    task = str(row["task"])
    fold = int(row["outer_fold"])

    outrow = {
        "task": task,
        "participant_id": str(row["participant_id"]),
        "logical_recording_id": str(row["logical_recording_id"]),
        "outer_fold": fold,
        "family": row["family"],
        "transform": row["transform"],
        "dose_label": row["dose_label"],
        "dose_code": int(row["dose_code"]),
        "candidate_value": row["candidate_value"],
        "candidate_unit": row["candidate_unit"],
        "exemplar": int(row["exemplar"]),
        "y": float(row["y"]),
    }

    target, q_target = response_row_to_inference_frames(row)

    for model_name in RIDGE_MODELS:
        state = hgb_states[(task, fold, model_name)]
        label = f"HGB_{model_name}"

        try:
            pred = float(
                bridge["predict_from_state"](
                    target,
                    state,
                    q_target,
                )[0]
            )
            outrow[f"prediction__{label}"] = pred
            outrow[f"prediction_status__{label}"] = "PASS"
            outrow[f"prediction_error__{label}"] = ""
        except Exception as exc:
            outrow[f"prediction__{label}"] = np.nan
            outrow[f"prediction_status__{label}"] = "UNAVAILABLE"
            outrow[f"prediction_error__{label}"] = (
                f"{type(exc).__name__}: {exc}"
            )

    hgb_response_rows.append(outrow)

    if i % 2000 == 0 or i == len(response):
        print(f"HGB inference: {i}/{len(response)} Stage-E rows")

hgb_response = pd.DataFrame(hgb_response_rows)

for label in HGB_MODELS:
    baseline_bad = hgb_response.loc[
        hgb_response["transform"].eq("baseline")
        & ~hgb_response[f"prediction_status__{label}"].eq("PASS")
    ]
    if len(baseline_bad):
        display(baseline_bad.head(20))
        raise RuntimeError(
            f"{label}: HGB prediction unavailable on unmodified baseline."
        )

atomic_csv(
    hgb_response,
    TABLES / "goal3_hgb_controlled_response_exemplar.csv",
)

hgb_availability = pd.DataFrame([
    {
        "model": label,
        "baseline_unavailable_rows": int(
            hgb_response.loc[
                hgb_response["transform"].eq("baseline"),
                f"prediction_status__{label}",
            ].ne("PASS").sum()
        ),
        "perturbed_unavailable_rows": int(
            hgb_response.loc[
                ~hgb_response["transform"].eq("baseline"),
                f"prediction_status__{label}",
            ].ne("PASS").sum()
        ),
        "perturbed_unavailable_participants": int(
            hgb_response.loc[
                ~hgb_response["transform"].eq("baseline")
                & hgb_response[f"prediction_status__{label}"].ne("PASS"),
                "participant_id",
            ].nunique()
        ),
    }
    for label in HGB_MODELS
])

atomic_csv(
    hgb_availability,
    TABLES / "goal3_hgb_prediction_availability.csv",
)

print("HGB PREDICTION AVAILABILITY")
display(hgb_availability)

HGB inference: 2000/14792 Stage-E rows
HGB inference: 4000/14792 Stage-E rows
HGB inference: 6000/14792 Stage-E rows
HGB inference: 8000/14792 Stage-E rows
HGB inference: 10000/14792 Stage-E rows
HGB inference: 12000/14792 Stage-E rows
HGB inference: 14000/14792 Stage-E rows
HGB inference: 14792/14792 Stage-E rows
HGB PREDICTION AVAILABILITY


,model,baseline_unavailable_rows,perturbed_unavailable_rows,perturbed_unavailable_participants
0,HGB_M_A,0,0,0
1,HGB_M_A+Q,0,191,15
2,HGB_M_A-resQ,0,191,15


## 8. HGB perturbation deltas, dose response, and paired HGB-versus-ridge bootstrap

The HGB sensitivity uses the same scientific outputs as the ridge primary:
diagnosis Δlogit and ΔBrier; severity Δprediction and Δabsolute error.

In [15]:
def safe_logit(p, eps=1e-8):
    p = float(np.clip(float(p), eps, 1 - eps))
    return math.log(p / (1 - p))

hgb_baseline = hgb_response.loc[
    hgb_response["transform"].eq("baseline")
].copy()

if hgb_baseline.groupby(["task", "participant_id"]).size().ne(1).any():
    raise RuntimeError("HGB baseline is not one row/task/participant.")

hgb_base_lookup = hgb_baseline.set_index(["task", "participant_id"])

delta_rows = []

for row in hgb_response.loc[
    ~hgb_response["transform"].eq("baseline")
].to_dict("records"):
    key = (row["task"], str(row["participant_id"]))
    b = hgb_base_lookup.loc[key]

    for label in HGB_MODELS:
        bp = pd.to_numeric(
            pd.Series([b.get(f"prediction__{label}", np.nan)]),
            errors="coerce",
        ).iloc[0]
        pp = pd.to_numeric(
            pd.Series([row.get(f"prediction__{label}", np.nan)]),
            errors="coerce",
        ).iloc[0]

        common = {
            "task": row["task"],
            "participant_id": str(row["participant_id"]),
            "logical_recording_id": str(row["logical_recording_id"]),
            "outer_fold": int(row["outer_fold"]),
            "family": row["family"],
            "transform": row["transform"],
            "dose_label": row["dose_label"],
            "dose_code": int(row["dose_code"]),
            "candidate_value": row["candidate_value"],
            "candidate_unit": row["candidate_unit"],
            "exemplar": int(row["exemplar"]),
            "model": label,
            "y": float(row["y"]),
            "baseline_prediction": bp,
            "perturbed_prediction": pp,
            "baseline_prediction_available": bool(np.isfinite(bp)),
            "perturbed_prediction_available": bool(np.isfinite(pp)),
        }

        if row["task"] == "diagnosis":
            if np.isfinite(bp) and np.isfinite(pp):
                bl = safe_logit(bp)
                pl = safe_logit(pp)
                be = (float(row["y"]) - bp) ** 2
                pe = (float(row["y"]) - pp) ** 2
            else:
                bl = pl = be = pe = np.nan

            delta_rows.append({
                **common,
                "baseline_logit": bl,
                "perturbed_logit": pl,
                "delta_logit": (
                    float(pl - bl)
                    if np.isfinite(bl) and np.isfinite(pl)
                    else np.nan
                ),
                "delta_probability": (
                    float(pp - bp)
                    if np.isfinite(bp) and np.isfinite(pp)
                    else np.nan
                ),
                "delta_prediction": np.nan,
                "baseline_error": be,
                "perturbed_error": pe,
                "delta_error": (
                    float(pe - be)
                    if np.isfinite(be) and np.isfinite(pe)
                    else np.nan
                ),
                "error_metric": "Brier contribution",
            })
        else:
            if np.isfinite(bp) and np.isfinite(pp):
                be = abs(float(row["y"]) - bp)
                pe = abs(float(row["y"]) - pp)
            else:
                be = pe = np.nan

            delta_rows.append({
                **common,
                "baseline_logit": np.nan,
                "perturbed_logit": np.nan,
                "delta_logit": np.nan,
                "delta_probability": np.nan,
                "delta_prediction": (
                    float(pp - bp)
                    if np.isfinite(bp) and np.isfinite(pp)
                    else np.nan
                ),
                "baseline_error": be,
                "perturbed_error": pe,
                "delta_error": (
                    float(pe - be)
                    if np.isfinite(be) and np.isfinite(pe)
                    else np.nan
                ),
                "error_metric": "absolute error",
            })

hgb_delta_exemplar = pd.DataFrame(delta_rows)

group_cols = [
    "task", "participant_id", "logical_recording_id", "outer_fold",
    "family", "transform", "dose_label", "dose_code",
    "candidate_value", "candidate_unit", "model", "y", "error_metric",
]

hgb_delta = (
    hgb_delta_exemplar.groupby(
        group_cols,
        dropna=False,
        as_index=False,
    )
    .agg(
        baseline_prediction=("baseline_prediction", "first"),
        perturbed_prediction=("perturbed_prediction", "mean"),
        baseline_logit=("baseline_logit", "first"),
        perturbed_logit=("perturbed_logit", "mean"),
        delta_logit=("delta_logit", "mean"),
        delta_probability=("delta_probability", "mean"),
        delta_prediction=("delta_prediction", "mean"),
        baseline_error=("baseline_error", "first"),
        perturbed_error=("perturbed_error", "mean"),
        delta_error=("delta_error", "mean"),
        n_exemplars=("exemplar", "size"),
        n_finite_predictions=(
            "perturbed_prediction",
            lambda s: int(np.isfinite(
                pd.to_numeric(s, errors="coerce")
            ).sum()),
        ),
    )
)

atomic_csv(
    hgb_delta_exemplar,
    TABLES / "goal3_hgb_prediction_delta_exemplar.csv",
)
atomic_csv(
    hgb_delta,
    TABLES / "goal3_hgb_prediction_delta.csv",
)

hgb_dose_rows = []

for task in TASKS:
    endpoint_list = endpoint_map[task]

    for transform in sorted(
        hgb_delta.loc[
            hgb_delta["task"].eq(task),
            "transform",
        ].dropna().unique()
    ):
        for label in HGB_MODELS:
            g = hgb_delta.loc[
                hgb_delta["task"].eq(task)
                & hgb_delta["transform"].eq(transform)
                & hgb_delta["model"].eq(label)
            ].copy()

            for endpoint in endpoint_list:
                local = g[
                    ["participant_id", "dose_code", endpoint]
                ].copy()

                base = (
                    local[["participant_id"]]
                    .drop_duplicates()
                    .copy()
                )
                base["dose_code"] = 0
                base[endpoint] = 0.0

                analysis = pd.concat(
                    [base, local],
                    ignore_index=True,
                )

                est, se, p, status = fit_gee_slope(
                    analysis,
                    endpoint,
                )

                hgb_dose_rows.append({
                    "task": task,
                    "transform": transform,
                    "model": label,
                    "endpoint": endpoint,
                    "estimate_per_dose": est,
                    "se": se,
                    "p_value": p,
                    "participants": analysis["participant_id"].nunique(),
                    "status": status,
                })

hgb_dose = pd.DataFrame(hgb_dose_rows)

if not hgb_dose["status"].eq("PASS").all():
    display(hgb_dose.loc[~hgb_dose["status"].eq("PASS")])
    raise RuntimeError("At least one HGB dose-response GEE failed.")

atomic_csv(
    hgb_dose,
    TABLES / "goal3_hgb_dose_response.csv",
)

def participant_slopes(frame, task, model, transform, endpoint):
    g = frame.loc[
        frame["task"].eq(task)
        & frame["model"].eq(model)
        & frame["transform"].eq(transform),
        ["participant_id", "dose_code", endpoint],
    ].copy()

    out = []
    for pid, z in g.groupby("participant_id"):
        z = z.copy()
        z[endpoint] = pd.to_numeric(z[endpoint], errors="coerce")
        z["dose_code"] = pd.to_numeric(z["dose_code"], errors="coerce")
        z = z.loc[
            np.isfinite(z[endpoint])
            & np.isfinite(z["dose_code"])
        ]

        # Add the known paired baseline point (0, 0).
        x = np.r_[0.0, z["dose_code"].to_numpy(float)]
        y = np.r_[0.0, z[endpoint].to_numpy(float)]

        if len(np.unique(x)) < 2:
            continue

        slope = float(
            np.sum((x - x.mean()) * (y - y.mean()))
            / np.sum((x - x.mean()) ** 2)
        )
        out.append({
            "participant_id": str(pid),
            "slope": slope,
        })

    return pd.DataFrame(out)

hgb_vs_ridge_rows = []

model_pairs = {
    "HGB_M_A": "M_A",
    "HGB_M_A+Q": "M_A+Q",
    "HGB_M_A-resQ": "M_A-resQ",
}

for task in TASKS:
    primary_endpoint = (
        "delta_logit" if task == "diagnosis" else "delta_prediction"
    )

    transforms = sorted(
        prediction_primary.loc[
            prediction_primary["task"].eq(task),
            "transform",
        ].dropna().unique()
    )

    for transform in transforms:
        for hgb_model, ridge_model in model_pairs.items():
            h = participant_slopes(
                hgb_delta,
                task,
                hgb_model,
                transform,
                primary_endpoint,
            ).rename(columns={"slope": "hgb_slope"})

            r = participant_slopes(
                prediction_primary,
                task,
                ridge_model,
                transform,
                primary_endpoint,
            ).rename(columns={"slope": "ridge_slope"})

            pair = h.merge(
                r,
                on="participant_id",
                how="inner",
                validate="one_to_one",
            )

            if len(pair) < 20:
                raise RuntimeError(
                    f"Too few paired HGB/ridge participants: "
                    f"{task}/{transform}/{hgb_model}"
                )

            delta = (
                pair["hgb_slope"].to_numpy(float)
                - pair["ridge_slope"].to_numpy(float)
            )
            estimate = float(np.mean(delta))

            rng = np.random.default_rng(
                deterministic_seed(
                    "goal3_hgb_vs_ridge_bootstrap",
                    task,
                    transform,
                    hgb_model,
                )
            )
            n = len(pair)
            draws = np.empty(N_BOOTSTRAPS, float)

            for b in range(N_BOOTSTRAPS):
                idx = rng.integers(0, n, size=n)
                draws[b] = float(np.mean(delta[idx]))

            hgb_vs_ridge_rows.append({
                "task": task,
                "transform": transform,
                "endpoint": primary_endpoint,
                "hgb_model": hgb_model,
                "ridge_model": ridge_model,
                "participants": n,
                "estimate_hgb_minus_ridge_slope": estimate,
                "ci_low": float(np.quantile(draws, 0.025)),
                "ci_high": float(np.quantile(draws, 0.975)),
                "n_bootstraps": N_BOOTSTRAPS,
                "same_mean_slope_direction": bool(
                    np.sign(pair["hgb_slope"].mean())
                    == np.sign(pair["ridge_slope"].mean())
                ),
                "status": "PASS",
            })

hgb_vs_ridge = pd.DataFrame(hgb_vs_ridge_rows)

if len(hgb_vs_ridge) != 36:
    raise RuntimeError(
        f"Expected 36 HGB-vs-ridge bootstrap rows; found {len(hgb_vs_ridge)}."
    )
if not hgb_vs_ridge["n_bootstraps"].eq(2000).all():
    raise RuntimeError("HGB-vs-ridge package is not uniformly B=2,000.")

atomic_csv(
    hgb_vs_ridge,
    TABLES / "goal3_hgb_vs_ridge_bootstrap.csv",
)

print("=" * 78)
print("FROZEN HGB PERTURBATION SENSITIVITY: PASS")
print("=" * 78)
print("HGB dose-response rows:", len(hgb_dose))
print("HGB-vs-ridge bootstrap rows:", len(hgb_vs_ridge))

FROZEN HGB PERTURBATION SENSITIVITY: PASS
HGB dose-response rows: 72
HGB-vs-ridge bootstrap rows: 36


## 9. Final Goal-3 robustness/computational completion gate

This cell does **not** freeze publication figures. It certifies that the
prespecified computational analysis is complete and creates the immutable source
package for the subsequent figure-only notebook.

In [16]:
completion_required = [
    STAGE_E / "GOAL3_STAGE_E_PRIMARY_SEAL.json",
    STAGE_E_TABLES / "goal3_controlled_response_exemplar.csv",
    STAGE_E_TABLES / "goal3_feature_delta.csv",
    STAGE_E_TABLES / "goal3_prediction_delta.csv",
    STAGE_E_TABLES / "goal3_target_q_verification.csv",
    STAGE_E_TABLES / "goal3_offtarget_q_summary.csv",
    STAGE_E_TABLES / "goal3_A_family_movement_summary.csv",
    STAGE_E_TABLES / "goal3_dose_response_tests.csv",
    STAGE_E_TABLES / "goal3_bootstrap_ci.csv",
    STAGE_E_TABLES / "goal3_qdist_baseline_negative_sensitivity.csv",
    STAGE_E_AUDIT / "goal3_target_q_order_audit.csv",
    TABLES / "goal3_prediction_availability_by_model.csv",
    TABLES / "goal3_prediction_unavailable_detail.csv",
    TABLES / "goal3_primary_clinical_dose_linear.csv",
    TABLES / "goal3_ordered_factor_sensitivity.csv",
    TABLES / "goal3_heterogeneity_sensitivity.csv",
    TABLES / "goal3_model_interaction_tests.csv",
    TABLES / "goal3_exemplar_level_clinical_sensitivity.csv",
    TABLES / "goal3_exemplar_level_targetQ_sensitivity.csv",
    TABLES / "goal3_segmentation_change_summary.csv",
    AUDIT / "goal3_fixed_baseline_segmentation_disposition.json",
    TABLES / "goal3_hgb_goal2_reproduction_audit.csv",
    TABLES / "goal3_hgb_prediction_availability.csv",
    TABLES / "goal3_hgb_prediction_delta.csv",
    TABLES / "goal3_hgb_dose_response.csv",
    TABLES / "goal3_hgb_vs_ridge_bootstrap.csv",
    hgb_state_path,
]

missing = [
    str(p) for p in completion_required
    if not Path(p).exists() or Path(p).stat().st_size == 0
]
if missing:
    raise RuntimeError(
        "Goal 3 cannot receive computational completion status. "
        "Missing/empty artifacts:\n" + "\n".join(missing)
    )

hgb_repro_max = float(
    hgb_reproduction["max_abs_prediction_difference"].max()
)
if hgb_repro_max > 1e-10:
    raise RuntimeError(
        f"HGB reproduction max difference exceeds tolerance: {hgb_repro_max}"
    )

artifact_hashes = {
    str(Path(p).relative_to(ROOT)): sha256_file(Path(p))
    for p in completion_required
}

completion_manifest = {
    "created_utc": utc_now(),
    "status": "PASS_PENDING_PUBLICATION_FIGURES",
    "goal": 3,
    "engine_version": ENGINE_VERSION,
    "stageE_response_rows": int(len(response)),
    "stageE_measurement_unavailable_rows": int(
        response["execution_status"].ne("PASS").sum()
    ),
    "target_q_all_strictly_ordered": True,
    "target_q_all_medians_expected_direction": True,
    "stageE_gee_rows": int(len(dose_tests)),
    "stageE_gee_all_pass": True,
    "stageE_bootstrap_replicates": 2000,
    "ridge_prediction_availability": prediction_availability.to_dict("records"),
    "exemplar_level_sensitivity": "PASS",
    "ordered_factor_sensitivity": "PASS",
    "qdist_baseline_negative_sensitivity": "PASS",
    "diagnosis_and_severity_heterogeneity_checks": "PASS",
    "fixed_baseline_segmentation_sensitivity": (
        "NOT_TECHNICALLY_FEASIBLE_FROM_FROZEN_STAGEE_ARTIFACTS"
    ),
    "fixed_baseline_segmentation_reason": (
        "Complete sample-level baseline interval masks were not persisted. "
        "No pseudo-fixed approximation was substituted."
    ),
    "segmentation_change_audit": "PASS",
    "hgb_model_form_sensitivity": "PASS",
    "hgb_goal2_reproduction_max_abs_diff": hgb_repro_max,
    "hgb_vs_ridge_bootstrap_replicates": 2000,
    "interpretation_boundary": (
        "Same-source dose-dependent changes support direct sensitivity only "
        "to the imposed digital transformation. Natural Q-A associations "
        "remain observational and do not identify physical cause."
    ),
    "next_step": (
        "Run the Goal-3 publication-figure-only notebook from these frozen "
        "machine-readable outputs, visually audit Figure 5/Figure 6 and "
        "supplementary panels, then write the final Goal-3 freeze seal."
    ),
    "artifact_hashes": artifact_hashes,
}

completion_manifest_path = OUT / "GOAL3_COMPLETION_MANIFEST.json"
atomic_json(completion_manifest, completion_manifest_path)

atomic_json(
    {
        "status": "PASS_PENDING_PUBLICATION_FIGURES",
        "goal": 3,
        "completion_manifest": str(completion_manifest_path.relative_to(ROOT)),
        "completion_manifest_sha256": sha256_file(completion_manifest_path),
    },
    OUT / "DONE_COMPUTATIONAL.json",
)

print("=" * 78)
print("GOAL 3 COMPUTATIONAL COMPLETION: PASS_PENDING_PUBLICATION_FIGURES")
print("=" * 78)
print("Completion manifest:", completion_manifest_path)
print("Stage-E waveform measurements rerun: NO")
print("Frozen HGB Goal-2 reproduction max |Δ|:", f"{hgb_repro_max:.3e}")
print("Next: figure-only publication notebook, visual review, final freeze.")

GOAL 3 COMPUTATIONAL COMPLETION: PASS_PENDING_PUBLICATION_FIGURES
Completion manifest: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\goal3_completion_v1_0\final\GOAL3_COMPLETION_MANIFEST.json
Stage-E waveform measurements rerun: NO
Frozen HGB Goal-2 reproduction max |Δ|: 1.776e-15
Next: figure-only publication notebook, visual review, final freeze.
